# Mean-Variance Optimization\n\n**Level:** Intermediate\n**Concepts Covered:** 6\n\n---\n\n## Overview\n\nThis comprehensive notebook covers mean-variance optimization with detailed explanations, mathematical derivations, Python implementations, and practical examples.

## 1. Introduction

### Why Mean-Variance Optimization Matters

**Mean-variance optimization (MVO)**, developed by Harry Markowitz in 1952, revolutionized portfolio management by providing a mathematical framework for diversification. It remains the cornerstone of modern portfolio theory (MPT) and is used by:

- **Pension funds** managing trillions in retirement assets
- **Robo-advisors** automating portfolio construction for retail investors
- **Asset managers** constructing balanced multi-asset portfolios
- **Risk managers** analyzing portfolio risk-return trade-offs
- **Financial planners** allocating client assets across asset classes

### Real-World Applications

- **401(k) Target-Date Funds**: Use MVO to adjust stock/bond allocation over time
- **Risk Parity Strategies**: Extension of MVO equalizing risk contributions
- **Strategic Asset Allocation**: Long-term policy portfolios for institutions
- **Multi-Asset Funds**: Optimal blending of stocks, bonds, commodities, alternatives
- **Currency Hedging**: Optimal hedge ratios for international portfolios

### Learning Objectives

By the end of this notebook, you will be able to:

1. **Calculate** portfolio expected return and variance from asset returns and covariances
2. **Derive** the efficient frontier analytically for two assets and numerically for N assets
3. **Implement** Markowitz optimization using scipy.optimize and cvxpy
4. **Identify** minimum variance portfolio and maximum Sharpe ratio (tangency) portfolio
5. **Apply** practical constraints (long-only, leverage limits, sector constraints)
6. **Evaluate** optimized portfolios and understand limitations of MVO

### Prerequisites

- Basic statistics (mean, variance, covariance, correlation)
- Linear algebra (matrix multiplication, quadratic forms)
- Python programming (NumPy, Pandas, Matplotlib)
- Understanding of risk and return concepts

### Historical Context

Harry Markowitz's 1952 paper "Portfolio Selection" introduced mean-variance optimization:

- **Key Insight**: Diversification reduces risk below the weighted average of individual asset risks
- **Nobel Prize**: Markowitz shared the 1990 Nobel Prize in Economics for this work
- **Industry Impact**: MVO became standard for institutional portfolio management by the 1970s
- **Modern Extensions**: Black-Litterman (1992), robust optimization, factor models

Despite criticisms (unstable inputs, unrealistic assumptions), MVO remains fundamental to portfolio theory.

**Estimated Time**: 90-120 minutes

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, optimize
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

# Set random seed for reproducibility
np.random.seed(42)

print('[OK] Libraries imported successfully')

## 2. Expected Return & Portfolio Mathematics

### Theory

The **portfolio expected return** is the weighted average of the expected returns of individual assets. This is one of the most fundamental concepts in portfolio theory.

**Key Concepts:**

1. **Linear Aggregation**: Portfolio return is a linear combination of asset returns
2. **Weight Constraint**: Weights must sum to 1 (or 100%)
3. **No Diversification Benefit for Returns**: Unlike variance, expected return doesn't benefit from diversification
4. **Long/Short Positions**: Positive weights = long positions, negative weights = short positions

**Why This Matters:**

- **Benchmark Construction**: Understanding how index returns aggregate from constituents
- **Performance Attribution**: Decomposing portfolio returns into asset contributions
- **Rebalancing Decisions**: Predicting impact of weight changes on expected return
- **Asset Allocation**: Setting strategic targets for different asset classes

**Mathematical Intuition:**

If you invest 60% in stocks (expected return 10%) and 40% in bonds (expected return 4%), your portfolio's expected return is:

$$R_p = 0.60 \times 10\% + 0.40 \times 4\% = 6\% + 1.6\% = 7.6\%$$

This linear relationship makes portfolio return calculation straightforward, unlike portfolio risk which involves covariances.

### Mathematical Formulation

The portfolio expected return is formally defined as:

$$
E[R_p] = \sum_{i=1}^{N} w_i \cdot E[R_i] = \mathbf{w}^T \boldsymbol{\mu}
$$

**Notation:**

- $E[R_p]$ = Expected return of the portfolio
- $N$ = Number of assets
- $w_i$ = Weight of asset $i$ in the portfolio
- $E[R_i]$ = Expected return of asset $i$
- $\mathbf{w}$ = Weight vector $(w_1, w_2, \ldots, w_N)^T$
- $\boldsymbol{\mu}$ = Expected return vector $(E[R_1], E[R_2], \ldots, E[R_N])^T$

**Constraints:**

1. **Full Investment Constraint** (long-only):
   $$\sum_{i=1}^{N} w_i = 1, \quad w_i \geq 0 \quad \forall i$$

2. **Long-Short Constraint** (market neutral example):
   $$\sum_{i=1}^{N} w_i = 0 \text{ or } 1$$

**Example - Three Asset Portfolio:**

For a portfolio with stocks, bonds, and commodities:

$$
E[R_p] = w_s \cdot \mu_s + w_b \cdot \mu_b + w_c \cdot \mu_c
$$

where weights satisfy: $w_s + w_b + w_c = 1$

**Vector Representation:**

$$
E[R_p] = \begin{bmatrix} w_s & w_b & w_c \end{bmatrix} \begin{bmatrix} \mu_s \\ \mu_b \\ \mu_c \end{bmatrix}
$$

This compact matrix notation becomes essential when working with many assets.

In [ ]:
# Python implementation for Portfolio Expected Return

def portfolio_expected_return(weights, expected_returns):
    """
    Calculate portfolio expected return.
    
    Parameters
    ----------
    weights : np.ndarray
        Array of portfolio weights, shape (N,)
    expected_returns : np.ndarray
        Array of asset expected returns, shape (N,)
    
    Returns
    -------
    float
        Portfolio expected return
    
    Notes
    -----
    Implements the formula:
        E[R_p] = w^T * mu
    
    Examples
    --------
    >>> weights = np.array([0.6, 0.4])
    >>> returns = np.array([0.10, 0.04])
    >>> portfolio_expected_return(weights, returns)
    0.076
    """
    return np.dot(weights, expected_returns)


def generate_sample_returns(expected_returns, volatilities, correlation_matrix, 
                           num_periods=252, annual=True):
    """
    Generate sample asset returns using multivariate normal distribution.
    
    Parameters
    ----------
    expected_returns : np.ndarray
        Array of expected annual returns, shape (N,)
    volatilities : np.ndarray
        Array of annual volatilities, shape (N,)
    correlation_matrix : np.ndarray
        Correlation matrix, shape (N, N)
    num_periods : int, optional
        Number of time periods to simulate (default: 252 trading days)
    annual : bool, optional
        If True, returns are annualized (default: True)
    
    Returns
    -------
    np.ndarray
        Simulated returns, shape (num_periods, N)
    
    Notes
    -----
    Converts annual parameters to period parameters:
        - Period return = Annual return / periods_per_year
        - Period vol = Annual vol / sqrt(periods_per_year)
    
    Examples
    --------
    >>> mu = np.array([0.10, 0.04])
    >>> sigma = np.array([0.20, 0.08])
    >>> corr = np.array([[1.0, 0.3], [0.3, 1.0]])
    >>> returns = generate_sample_returns(mu, sigma, corr, 252)
    """
    n_assets = len(expected_returns)
    
    # Convert annual parameters to period parameters
    if annual:
        periods_per_year = num_periods
        period_returns = expected_returns / periods_per_year
        period_vols = volatilities / np.sqrt(periods_per_year)
    else:
        period_returns = expected_returns
        period_vols = volatilities
    
    # Construct covariance matrix from correlation and volatilities
    # Cov = D * Corr * D, where D is diagonal matrix of volatilities
    D = np.diag(period_vols)
    cov_matrix = D @ correlation_matrix @ D
    
    # Generate multivariate normal samples
    returns = np.random.multivariate_normal(period_returns, cov_matrix, num_periods)
    
    return returns


def portfolio_returns_over_time(weights, asset_returns):
    """
    Calculate portfolio returns over time given asset returns.
    
    Parameters
    ----------
    weights : np.ndarray
        Portfolio weights, shape (N,)
    asset_returns : np.ndarray
        Asset returns over time, shape (T, N)
    
    Returns
    -------
    np.ndarray
        Portfolio returns over time, shape (T,)
    
    Notes
    -----
    For each time period t:
        R_p(t) = sum(w_i * R_i(t))
    
    Examples
    --------
    >>> weights = np.array([0.6, 0.4])
    >>> returns = np.array([[0.01, 0.005], [0.02, 0.008]])
    >>> portfolio_returns_over_time(weights, returns)
    array([0.008, 0.0152])
    """
    return asset_returns @ weights


# Test the implementation
print("[OK] Portfolio expected return functions implemented")
print("\nFunction capabilities:")
print("  - portfolio_expected_return(): Calculate E[R_p] = w^T * mu")
print("  - generate_sample_returns(): Simulate asset returns with correlation")
print("  - portfolio_returns_over_time(): Compute portfolio returns series")

In [ ]:
# Visualization for Portfolio Expected Return - 3 Asset Example

# Define 3-asset portfolio parameters
asset_names = ['US Stocks', 'Treasury Bonds', 'Gold']
expected_returns = np.array([0.10, 0.04, 0.06])  # 10%, 4%, 6% annual
volatilities = np.array([0.18, 0.07, 0.20])      # 18%, 7%, 20% annual

# Correlation matrix (realistic values)
correlation_matrix = np.array([
    [1.00, 0.10, 0.05],  # Stocks: low correlation with bonds/gold
    [0.10, 1.00, -0.15], # Bonds: negative correlation with gold
    [0.05, -0.15, 1.00]  # Gold: flight to safety asset
])

# Portfolio weights
weights = np.array([0.50, 0.35, 0.15])  # 50% stocks, 35% bonds, 15% gold

# Generate sample returns (1 year = 252 trading days)
np.random.seed(42)
asset_returns = generate_sample_returns(expected_returns, volatilities, 
                                       correlation_matrix, num_periods=252)

# Calculate portfolio returns over time
portfolio_returns = portfolio_returns_over_time(weights, asset_returns)

# Calculate cumulative returns
cum_asset_returns = (1 + asset_returns).cumprod(axis=0)
cum_portfolio_returns = (1 + portfolio_returns).cumprod()

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Cumulative Returns Comparison
ax = axes[0, 0]
for i, name in enumerate(asset_names):
    ax.plot(cum_asset_returns[:, i], label=name, linewidth=2, alpha=0.7)
ax.plot(cum_portfolio_returns, label='Portfolio', linewidth=2.5, 
        color='black', linestyle='--')
ax.set_title('Cumulative Returns: Individual Assets vs Portfolio', fontsize=12, fontweight='bold')
ax.set_xlabel('Trading Days')
ax.set_ylabel('Cumulative Return (Initial = 1.0)')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
ax.axhline(y=1.0, color='gray', linestyle=':', alpha=0.5)

# Plot 2: Return Distribution Comparison
ax = axes[0, 1]
colors = plt.cm.Set3(range(len(asset_names)))
for i, name in enumerate(asset_names):
    ax.hist(asset_returns[:, i], bins=30, alpha=0.5, label=name, color=colors[i])
ax.hist(portfolio_returns, bins=30, alpha=0.7, label='Portfolio', 
        color='black', edgecolor='black', linewidth=1.5)
ax.set_title('Return Distribution Comparison', fontsize=12, fontweight='bold')
ax.set_xlabel('Daily Returns')
ax.set_ylabel('Frequency')
ax.legend(loc='best')
ax.grid(True, alpha=0.3, axis='y')

# Plot 3: Portfolio Weight Contribution
ax = axes[1, 0]
contribution = weights * expected_returns
colors_pie = plt.cm.Set2(range(len(asset_names)))
wedges, texts, autotexts = ax.pie(contribution, labels=asset_names, autopct='%1.1f%%',
                                    colors=colors_pie, startangle=90)
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
ax.set_title('Contribution to Portfolio Expected Return', fontsize=12, fontweight='bold')

# Plot 4: Expected Return Calculation Breakdown
ax = axes[1, 1]
x_pos = np.arange(len(asset_names))
ax.bar(x_pos, expected_returns * 100, alpha=0.6, label='Expected Return (%)', color='steelblue')
ax2 = ax.twinx()
ax2.bar(x_pos, weights * 100, alpha=0.6, label='Portfolio Weight (%)', color='coral')
ax.set_xticks(x_pos)
ax.set_xticklabels(asset_names, rotation=15)
ax.set_ylabel('Expected Return (%)', color='steelblue')
ax2.set_ylabel('Portfolio Weight (%)', color='coral')
ax.set_title('Portfolio Composition & Expected Returns', fontsize=12, fontweight='bold')
ax.tick_params(axis='y', labelcolor='steelblue')
ax2.tick_params(axis='y', labelcolor='coral')
ax.grid(True, alpha=0.3, axis='y')

# Add legend
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.show()

# Print summary statistics
print("\n" + "="*60)
print("PORTFOLIO EXPECTED RETURN ANALYSIS")
print("="*60)
print(f"\nAsset Allocation:")
for i, name in enumerate(asset_names):
    print(f"  {name:15s}: {weights[i]*100:5.1f}%")

print(f"\nExpected Returns (Annual):")
for i, name in enumerate(asset_names):
    print(f"  {name:15s}: {expected_returns[i]*100:5.1f}%")

print(f"\nContribution to Portfolio Return:")
for i, name in enumerate(asset_names):
    contrib = weights[i] * expected_returns[i]
    print(f"  {name:15s}: {contrib*100:5.2f}% (weight {weights[i]:.1%} × return {expected_returns[i]:.1%})")

portfolio_exp_return = portfolio_expected_return(weights, expected_returns)
print(f"\nPortfolio Expected Return: {portfolio_exp_return*100:.2f}%")
print(f"\nRealized Return (1-year simulation): {(cum_portfolio_returns[-1] - 1)*100:.2f}%")
print("\nNote: Realized return differs from expected due to randomness in simulation")

### Practical Example

Let's apply portfolio expected return calculations to a realistic scenario: comparing a conservative 60/40 portfolio with an aggressive 80/20 portfolio.

**Scenario:**
- Asset 1: S&P 500 ETF (SPY) with E[R] = 10% annual
- Asset 2: 10-Year Treasury ETF (IEF) with E[R] = 4% annual

**Question:** How much additional expected return do we gain by moving from a 60/40 to an 80/20 allocation?

In [ ]:
# Practical example: 60/40 vs 80/20 Portfolio Comparison

# Define asset parameters
asset_names_2 = ['S&P 500 (SPY)', '10-Yr Treasury (IEF)']
expected_returns_2 = np.array([0.10, 0.04])  # 10%, 4%

# Define portfolio allocations
portfolios = {
    'Conservative (60/40)': np.array([0.60, 0.40]),
    'Balanced (70/30)': np.array([0.70, 0.30]),
    'Aggressive (80/20)': np.array([0.80, 0.20]),
    'Very Aggressive (90/10)': np.array([0.90, 0.10])
}

# Calculate expected returns for each portfolio
print("="*70)
print("PORTFOLIO ALLOCATION COMPARISON")
print("="*70)

results = {}
for name, weights in portfolios.items():
    exp_return = portfolio_expected_return(weights, expected_returns_2)
    results[name] = exp_return
    
    print(f"\n{name}:")
    print(f"  Allocation: {weights[0]*100:.0f}% Stocks / {weights[1]*100:.0f}% Bonds")
    print(f"  Expected Return: {exp_return*100:.2f}%")
    print(f"  Calculation: {weights[0]:.1%} × {expected_returns_2[0]:.1%} + "
          f"{weights[1]:.1%} × {expected_returns_2[1]:.1%} = {exp_return:.2%}")

# Calculate incremental returns
print("\n" + "="*70)
print("INCREMENTAL RETURN ANALYSIS")
print("="*70)

conservative_return = results['Conservative (60/40)']
for name, exp_return in results.items():
    if name != 'Conservative (60/40)':
        incremental = exp_return - conservative_return
        incremental_bps = incremental * 10000
        print(f"\n{name} vs Conservative:")
        print(f"  Additional Expected Return: {incremental*100:.2f}% ({incremental_bps:.0f} bps)")
        print(f"  On $100,000 investment: ${incremental*100000:,.2f} additional per year")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Expected Return vs Stock Allocation
ax = axes[0]
stock_weights = np.array([p[0] for p in portfolios.values()])
expected_rets = np.array(list(results.values()))
ax.plot(stock_weights * 100, expected_rets * 100, 'o-', markersize=10, 
        linewidth=2, color='steelblue', label='Expected Return')
ax.fill_between(stock_weights * 100, expected_returns_2[1] * 100, 
                expected_rets * 100, alpha=0.3, color='steelblue')

# Annotate points
for name, weight, ret in zip(portfolios.keys(), stock_weights, expected_rets):
    label = name.split('(')[0].strip()
    ax.annotate(f'{label}\n{ret*100:.1f}%', 
               xy=(weight*100, ret*100), 
               xytext=(5, 5), textcoords='offset points',
               fontsize=9, bbox=dict(boxstyle='round,pad=0.3', 
                                     facecolor='yellow', alpha=0.3))

ax.set_title('Expected Return vs Stock Allocation', fontsize=12, fontweight='bold')
ax.set_xlabel('Stock Allocation (%)')
ax.set_ylabel('Expected Annual Return (%)')
ax.grid(True, alpha=0.3)
ax.axhline(y=expected_returns_2[0]*100, color='red', linestyle='--', 
          alpha=0.5, label='100% Stocks')
ax.axhline(y=expected_returns_2[1]*100, color='green', linestyle='--', 
          alpha=0.5, label='100% Bonds')
ax.legend(loc='best')

# Plot 2: Return Contribution Breakdown
ax = axes[1]
portfolio_names = list(portfolios.keys())
stock_contributions = []
bond_contributions = []

for weights in portfolios.values():
    stock_contrib = weights[0] * expected_returns_2[0] * 100
    bond_contrib = weights[1] * expected_returns_2[1] * 100
    stock_contributions.append(stock_contrib)
    bond_contributions.append(bond_contrib)

x_pos = np.arange(len(portfolio_names))
width = 0.6

p1 = ax.bar(x_pos, stock_contributions, width, label='Stock Contribution', 
           color='steelblue', alpha=0.8)
p2 = ax.bar(x_pos, bond_contributions, width, bottom=stock_contributions,
           label='Bond Contribution', color='coral', alpha=0.8)

# Add value labels
for i, (s, b) in enumerate(zip(stock_contributions, bond_contributions)):
    total = s + b
    ax.text(i, total + 0.1, f'{total:.1f}%', ha='center', fontweight='bold')
    ax.text(i, s/2, f'{s:.1f}%', ha='center', va='center', color='white', fontweight='bold')
    ax.text(i, s + b/2, f'{b:.1f}%', ha='center', va='center', color='white', fontweight='bold')

ax.set_title('Return Contribution Breakdown', fontsize=12, fontweight='bold')
ax.set_ylabel('Expected Return (%)')
ax.set_xticks(x_pos)
ax.set_xticklabels([p.split('(')[0].strip() for p in portfolio_names], rotation=15, ha='right')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Key insight
print("\n" + "="*70)
print("KEY INSIGHT")
print("="*70)
print("\nThe expected return increases LINEARLY with stock allocation because:")
print("  E[R_p] = w_stocks × E[R_stocks] + (1 - w_stocks) × E[R_bonds]")
print("\nThis is a straight line with:")
print(f"  - Slope = {(expected_returns_2[0] - expected_returns_2[1])*100:.1f}% "
      f"(difference in expected returns)")
print(f"  - Intercept = {expected_returns_2[1]*100:.1f}% (100% bonds)")
print(f"  - Max value = {expected_returns_2[0]*100:.1f}% (100% stocks)")
print("\nUnlike return, portfolio RISK does NOT scale linearly due to diversification!")
print("We'll explore this next in the covariance section.")

## 3. Covariance Matrix & Portfolio Variance

### Theory

The **covariance matrix** is the heart of portfolio risk analysis. Unlike expected return (which is simply a weighted average), portfolio variance depends on:

1. Individual asset variances (diagonal elements)
2. Pairwise covariances between assets (off-diagonal elements)

**Why Covariance Matters:**

- **Diversification Benefit**: Low or negative correlation reduces portfolio risk below weighted average
- **Risk Attribution**: Identifies which assets contribute most to portfolio variance
- **Optimal Allocation**: The efficient frontier depends critically on the covariance structure
- **Hedging**: Negative covariance assets provide natural hedges

**Key Insights:**

1. **Perfect Positive Correlation (ρ = +1)**: No diversification benefit, portfolio vol = weighted average vol
2. **Zero Correlation (ρ = 0)**: Significant diversification, portfolio vol < weighted average vol
3. **Negative Correlation (ρ < 0)**: Maximum diversification, can create low-risk portfolios
4. **Correlation Instability**: Correlations tend to increase during market stress ("correlation goes to 1 in a crisis")

**Real-World Examples:**

- **Stocks & Bonds**: Historically low correlation (ρ ≈ 0.1-0.3), providing diversification
- **Stocks & Gold**: Low/negative correlation (ρ ≈ -0.1 to 0.2), gold as hedge asset
- **Equity Sectors**: Within equities, correlations typically high (ρ ≈ 0.6-0.9)
- **International Equities**: Globalization increased correlations from ρ ≈ 0.4 (1980s) to ρ ≈ 0.8 (2020s)

**The Diversification Equation:**

For a 2-asset portfolio, variance is:

$$\sigma_p^2 = w_1^2\sigma_1^2 + w_2^2\sigma_2^2 + 2w_1w_2\sigma_1\sigma_2\rho_{12}$$

The last term is the **covariance term** - when ρ < 1, portfolio variance is reduced!

### Mathematical Formulation

The portfolio variance is a **quadratic form** involving the covariance matrix:

$$
\sigma_p^2 = \mathbf{w}^T \boldsymbol{\Sigma} \mathbf{w} = \sum_{i=1}^{N} \sum_{j=1}^{N} w_i w_j \sigma_{ij}
$$

**Notation:**

- $\sigma_p^2$ = Portfolio variance
- $\sigma_p$ = Portfolio volatility (standard deviation) = $\sqrt{\sigma_p^2}$
- $\mathbf{w}$ = Weight vector $(w_1, w_2, \ldots, w_N)^T$
- $\boldsymbol{\Sigma}$ = Covariance matrix (N × N symmetric matrix)
- $\sigma_{ij}$ = Covariance between asset i and asset j

**Covariance Matrix Structure:**

$$
\boldsymbol{\Sigma} = \begin{bmatrix}
\sigma_1^2 & \sigma_{12} & \cdots & \sigma_{1N} \\
\sigma_{12} & \sigma_2^2 & \cdots & \sigma_{2N} \\
\vdots & \vdots & \ddots & \vdots \\
\sigma_{1N} & \sigma_{2N} & \cdots & \sigma_N^2
\end{bmatrix}
$$

where:
- **Diagonal elements**: $\sigma_i^2$ = variance of asset i
- **Off-diagonal elements**: $\sigma_{ij} = \rho_{ij}\sigma_i\sigma_j$ = covariance between assets i and j
- **Symmetry**: $\sigma_{ij} = \sigma_{ji}$ (covariance matrix is symmetric)

**Relationship: Covariance, Correlation, and Volatility:**

$$
\sigma_{ij} = \rho_{ij} \cdot \sigma_i \cdot \sigma_j
$$

where $\rho_{ij} \in [-1, 1]$ is the correlation coefficient.

**Two-Asset Portfolio Variance (Expanded Form):**

$$
\sigma_p^2 = w_1^2\sigma_1^2 + w_2^2\sigma_2^2 + 2w_1w_2\rho_{12}\sigma_1\sigma_2
$$

This shows three components:
1. Contribution from asset 1: $w_1^2\sigma_1^2$
2. Contribution from asset 2: $w_2^2\sigma_2^2$
3. Interaction term: $2w_1w_2\rho_{12}\sigma_1\sigma_2$

**Matrix Form (2-Asset):**

$$
\sigma_p^2 = \begin{bmatrix} w_1 & w_2 \end{bmatrix}
\begin{bmatrix}
\sigma_1^2 & \rho_{12}\sigma_1\sigma_2 \\
\rho_{12}\sigma_1\sigma_2 & \sigma_2^2
\end{bmatrix}
\begin{bmatrix} w_1 \\ w_2 \end{bmatrix}
$$

In [ ]:
# Python implementation for Covariance Matrix and Portfolio Variance

def covariance_from_correlation(volatilities, correlation_matrix):
    """
    Construct covariance matrix from volatilities and correlation matrix.
    
    Parameters
    ----------
    volatilities : np.ndarray
        Array of asset volatilities (standard deviations), shape (N,)
    correlation_matrix : np.ndarray
        Correlation matrix, shape (N, N)
    
    Returns
    -------
    np.ndarray
        Covariance matrix, shape (N, N)
    
    Notes
    -----
    Implements the formula:
        Sigma = D * Corr * D
    where D is a diagonal matrix of volatilities.
    
    Alternatively, element-wise:
        Sigma[i,j] = rho[i,j] * sigma[i] * sigma[j]
    
    Examples
    --------
    >>> vols = np.array([0.20, 0.08])
    >>> corr = np.array([[1.0, 0.3], [0.3, 1.0]])
    >>> covariance_from_correlation(vols, corr)
    array([[0.04  , 0.0048],
           [0.0048, 0.0064]])
    """
    D = np.diag(volatilities)
    return D @ correlation_matrix @ D


def portfolio_variance(weights, covariance_matrix):
    """
    Calculate portfolio variance.
    
    Parameters
    ----------
    weights : np.ndarray
        Portfolio weights, shape (N,)
    covariance_matrix : np.ndarray
        Covariance matrix, shape (N, N)
    
    Returns
    -------
    float
        Portfolio variance
    
    Notes
    -----
    Implements the quadratic form:
        sigma_p^2 = w^T * Sigma * w
    
    Examples
    --------
    >>> weights = np.array([0.6, 0.4])
    >>> cov = np.array([[0.04, 0.0048], [0.0048, 0.0064]])
    >>> portfolio_variance(weights, cov)
    0.016896
    """
    return weights @ covariance_matrix @ weights


def portfolio_volatility(weights, covariance_matrix):
    """
    Calculate portfolio volatility (standard deviation).
    
    Parameters
    ----------
    weights : np.ndarray
        Portfolio weights, shape (N,)
    covariance_matrix : np.ndarray
        Covariance matrix, shape (N, N)
    
    Returns
    -------
    float
        Portfolio volatility (standard deviation)
    
    Notes
    -----
    Volatility is the square root of variance:
        sigma_p = sqrt(w^T * Sigma * w)
    
    Examples
    --------
    >>> weights = np.array([0.6, 0.4])
    >>> cov = np.array([[0.04, 0.0048], [0.0048, 0.0064]])
    >>> portfolio_volatility(weights, cov)
    0.13
    """
    return np.sqrt(portfolio_variance(weights, covariance_matrix))


def variance_decomposition(weights, covariance_matrix):
    """
    Decompose portfolio variance into individual and interaction components.
    
    Parameters
    ----------
    weights : np.ndarray
        Portfolio weights, shape (N,)
    covariance_matrix : np.ndarray
        Covariance matrix, shape (N, N)
    
    Returns
    -------
    dict
        Dictionary containing:
        - 'individual': Individual variance contributions, shape (N,)
        - 'interaction': Interaction term contribution
        - 'total': Total portfolio variance
    
    Notes
    -----
    For 2-asset portfolio:
        Individual[0] = w1^2 * sigma1^2
        Individual[1] = w2^2 * sigma2^2
        Interaction = 2 * w1 * w2 * rho12 * sigma1 * sigma2
    
    Examples
    --------
    >>> weights = np.array([0.6, 0.4])
    >>> cov = np.array([[0.04, 0.0048], [0.0048, 0.0064]])
    >>> variance_decomposition(weights, cov)
    """
    n = len(weights)
    individual = np.zeros(n)
    
    # Individual variance contributions (diagonal terms)
    for i in range(n):
        individual[i] = weights[i]**2 * covariance_matrix[i, i]
    
    # Interaction term (off-diagonal terms)
    interaction = portfolio_variance(weights, covariance_matrix) - individual.sum()
    
    return {
        'individual': individual,
        'interaction': interaction,
        'total': portfolio_variance(weights, covariance_matrix)
    }


def diversification_benefit(weights, volatilities, correlation_matrix):
    """
    Calculate diversification benefit as reduction in volatility.
    
    Parameters
    ----------
    weights : np.ndarray
        Portfolio weights, shape (N,)
    volatilities : np.ndarray
        Asset volatilities, shape (N,)
    correlation_matrix : np.ndarray
        Correlation matrix, shape (N, N)
    
    Returns
    -------
    dict
        Dictionary containing:
        - 'portfolio_vol': Actual portfolio volatility
        - 'weighted_avg_vol': Weighted average volatility (no diversification)
        - 'benefit': Reduction in volatility due to diversification
        - 'benefit_pct': Percentage reduction
    
    Notes
    -----
    Diversification benefit measures how much correlation < 1 reduces risk.
    Weighted average vol assumes perfect correlation (no diversification).
    
    Examples
    --------
    >>> weights = np.array([0.6, 0.4])
    >>> vols = np.array([0.20, 0.08])
    >>> corr = np.array([[1.0, 0.3], [0.3, 1.0]])
    >>> diversification_benefit(weights, vols, corr)
    """
    # Actual portfolio volatility with correlations
    cov_matrix = covariance_from_correlation(volatilities, correlation_matrix)
    actual_vol = portfolio_volatility(weights, cov_matrix)
    
    # Weighted average volatility (assumes perfect correlation)
    weighted_avg_vol = np.dot(weights, volatilities)
    
    # Diversification benefit
    benefit = weighted_avg_vol - actual_vol
    benefit_pct = (benefit / weighted_avg_vol) * 100 if weighted_avg_vol > 0 else 0
    
    return {
        'portfolio_vol': actual_vol,
        'weighted_avg_vol': weighted_avg_vol,
        'benefit': benefit,
        'benefit_pct': benefit_pct
    }


print("[OK] Covariance and portfolio variance functions implemented")
print("\nFunction capabilities:")
print("  - covariance_from_correlation(): Build Sigma from volatilities and correlations")
print("  - portfolio_variance(): Calculate sigma_p^2 = w^T * Sigma * w")
print("  - portfolio_volatility(): Calculate sigma_p = sqrt(variance)")
print("  - variance_decomposition(): Break down variance into components")
print("  - diversification_benefit(): Measure risk reduction from diversification")

In [ ]:
# Visualization for Covariance Matrix Analysis

# Define parameters for Tech vs Bonds portfolio
asset_names_cov = ['Tech Stocks (QQQ)', '10-Yr Treasury (IEF)']
volatilities_cov = np.array([0.25, 0.08])  # 25% tech vol, 8% bond vol
weights_cov = np.array([0.60, 0.40])        # 60/40 allocation

# Test different correlation scenarios
correlations_to_test = [-0.5, 0.0, 0.3, 0.9]

# Create 2x2 subplot
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Plot 1: Correlation Heatmaps
ax = axes[0, 0]
correlation_matrix_example = np.array([[1.0, 0.3], [0.3, 1.0]])
cov_matrix_example = covariance_from_correlation(volatilities_cov, correlation_matrix_example)

# Create combined visualization showing both correlation and covariance
combined_data = np.array([
    [f"ρ={correlation_matrix_example[0,0]:.1f}\nσ²={cov_matrix_example[0,0]:.4f}", 
     f"ρ={correlation_matrix_example[0,1]:.1f}\nCov={cov_matrix_example[0,1]:.4f}"],
    [f"ρ={correlation_matrix_example[1,0]:.1f}\nCov={cov_matrix_example[1,0]:.4f}", 
     f"ρ={correlation_matrix_example[1,1]:.1f}\nσ²={cov_matrix_example[1,1]:.4f}"]
])

im = ax.imshow(cov_matrix_example, cmap='RdYlGn_r', aspect='auto')
for i in range(2):
    for j in range(2):
        text = ax.text(j, i, combined_data[i, j], ha="center", va="center", 
                      color="black", fontsize=10, fontweight='bold',
                      bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(asset_names_cov, fontsize=9)
ax.set_yticklabels(asset_names_cov, fontsize=9)
ax.set_title('Covariance Matrix Structure\n(ρ = 0.3 correlation)', 
            fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax, label='Covariance')

# Plot 2: Variance Decomposition
ax = axes[0, 1]
corr_matrix_decomp = np.array([[1.0, 0.3], [0.3, 1.0]])
cov_matrix_decomp = covariance_from_correlation(volatilities_cov, corr_matrix_decomp)
decomp = variance_decomposition(weights_cov, cov_matrix_decomp)

components = ['Tech\nIndividual', 'Bond\nIndividual', 'Interaction\nTerm']
values = [decomp['individual'][0], decomp['individual'][1], decomp['interaction']]
colors_decomp = ['steelblue', 'coral', 'lightgreen']

bars = ax.bar(components, values, color=colors_decomp, alpha=0.7, edgecolor='black', linewidth=2)
for i, (bar, val) in enumerate(zip(bars, values)):
    height = bar.get_height()
    sign = '+' if val >= 0 else ''
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.001,
           f'{sign}{val:.4f}', ha='center', va='bottom', fontweight='bold')

ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.set_ylabel('Variance Contribution')
ax.set_title(f'Portfolio Variance Decomposition\nTotal Variance = {decomp["total"]:.4f}', 
            fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Plot 3: Diversification Benefit vs Correlation
ax = axes[1, 0]
benefits = []
portfolio_vols = []
weighted_vols = []

for corr in np.linspace(-1, 1, 50):
    corr_mat = np.array([[1.0, corr], [corr, 1.0]])
    div_ben = diversification_benefit(weights_cov, volatilities_cov, corr_mat)
    benefits.append(div_ben['benefit_pct'])
    portfolio_vols.append(div_ben['portfolio_vol'] * 100)
    weighted_vols.append(div_ben['weighted_avg_vol'] * 100)

corr_range = np.linspace(-1, 1, 50)
ax.plot(corr_range, benefits, linewidth=2.5, color='green', label='Diversification Benefit')
ax.fill_between(corr_range, 0, benefits, alpha=0.3, color='green')

# Mark key points
for test_corr in [-1, 0, 1]:
    idx = np.argmin(np.abs(corr_range - test_corr))
    ax.plot(test_corr, benefits[idx], 'ro', markersize=10)
    ax.annotate(f'ρ={test_corr:.0f}\n{benefits[idx]:.1f}%', 
               xy=(test_corr, benefits[idx]), xytext=(10, 10),
               textcoords='offset points', fontsize=9,
               bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.7),
               arrowprops=dict(arrowstyle='->', color='red'))

ax.set_xlabel('Correlation Coefficient (ρ)')
ax.set_ylabel('Diversification Benefit (%)')
ax.set_title('Diversification Benefit vs Correlation\n(60/40 Tech/Bond Portfolio)', 
            fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_xlim(-1, 1)
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
ax.legend(loc='best')

# Plot 4: Efficient Set for Two Assets with Different Correlations
ax = axes[1, 1]

for corr in correlations_to_test:
    corr_mat = np.array([[1.0, corr], [corr, 1.0]])
    cov_mat = covariance_from_correlation(volatilities_cov, corr_mat)
    
    # Vary weight from 0 to 1
    weight_range = np.linspace(0, 1, 100)
    vols = []
    
    for w in weight_range:
        weights_temp = np.array([w, 1-w])
        vol = portfolio_volatility(weights_temp, cov_mat)
        vols.append(vol * 100)
    
    ax.plot(vols, weight_range * 100, linewidth=2, 
           label=f'ρ = {corr:+.1f}', alpha=0.8)

# Add individual assets
ax.plot(volatilities_cov[0] * 100, 100, 'ro', markersize=12, 
       label=asset_names_cov[0], zorder=5)
ax.plot(volatilities_cov[1] * 100, 0, 'go', markersize=12, 
       label=asset_names_cov[1], zorder=5)

# Weighted average line (perfect correlation)
ax.plot([volatilities_cov[1]*100, volatilities_cov[0]*100], [0, 100], 
       'k--', linewidth=2, alpha=0.5, label='ρ = +1.0 (No diversification)')

ax.set_xlabel('Portfolio Volatility (%)')
ax.set_ylabel('Tech Stock Allocation (%)')
ax.set_title('Efficient Set: Impact of Correlation\n(Two-Asset Combinations)', 
            fontsize=12, fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary analysis
print("\n" + "="*80)
print("COVARIANCE MATRIX & DIVERSIFICATION ANALYSIS")
print("="*80)

print(f"\nPortfolio: {weights_cov[0]:.0%} {asset_names_cov[0]}, {weights_cov[1]:.0%} {asset_names_cov[1]}")
print(f"Asset Volatilities: {volatilities_cov[0]:.1%} (Tech), {volatilities_cov[1]:.1%} (Bonds)")

print("\n" + "-"*80)
print("CORRELATION SCENARIO ANALYSIS")
print("-"*80)

for test_corr in correlations_to_test:
    corr_mat = np.array([[1.0, test_corr], [test_corr, 1.0]])
    cov_mat = covariance_from_correlation(volatilities_cov, corr_mat)
    div_ben = diversification_benefit(weights_cov, volatilities_cov, corr_mat)
    decomp = variance_decomposition(weights_cov, cov_mat)
    
    print(f"\nCorrelation ρ = {test_corr:+.1f}:")
    print(f"  Portfolio Volatility:     {div_ben['portfolio_vol']*100:6.2f}%")
    print(f"  Weighted Avg Volatility:  {div_ben['weighted_avg_vol']*100:6.2f}%")
    print(f"  Diversification Benefit:  {div_ben['benefit']*100:6.2f}% ({div_ben['benefit_pct']:.1f}% reduction)")
    print(f"  Variance Decomposition:")
    print(f"    - Tech individual:      {decomp['individual'][0]:8.5f}")
    print(f"    - Bond individual:      {decomp['individual'][1]:8.5f}")
    print(f"    - Interaction term:     {decomp['interaction']:8.5f}")
    print(f"    - Total variance:       {decomp['total']:8.5f}")

print("\n" + "="*80)
print("KEY INSIGHT")
print("="*80)
print("\nDiversification benefit is MAXIMIZED when correlation is LOW or NEGATIVE:")
print("  - At ρ = -0.5: Maximum risk reduction, can nearly eliminate volatility")
print("  - At ρ =  0.0: Significant benefit from independence")
print("  - At ρ = +0.3: Moderate benefit (typical stocks/bonds)")
print("  - At ρ = +0.9: Minimal benefit (similar assets)")
print("  - At ρ = +1.0: NO benefit (perfectly correlated)")
print("\nThis is why portfolios combine assets with different risk drivers!")

### Practical Example

Let's examine a realistic portfolio decision: Should we add Gold to a traditional 60/40 Stocks/Bonds portfolio?

**Scenario:**
- Current Portfolio: 60% S&P 500, 40% 10-Year Treasuries
- Proposed Addition: Include 15% Gold allocation (reduce stocks to 45%, bonds to 40%)

**Question:** How does Gold's low correlation with stocks and bonds affect overall portfolio risk?

In [ ]:
# Practical example: Adding Gold to 60/40 Portfolio

# Define 3-asset parameters
asset_names_gold = ['S&P 500', '10-Yr Treasury', 'Gold']
expected_returns_gold = np.array([0.10, 0.04, 0.06])  # 10%, 4%, 6%
volatilities_gold = np.array([0.18, 0.07, 0.20])       # 18%, 7%, 20%

# Realistic correlation matrix
# Stocks-Bonds: +0.1 (low positive)
# Stocks-Gold: +0.05 (very low, sometimes negative)
# Bonds-Gold: -0.15 (negative, gold as flight-to-safety)
correlation_matrix_gold = np.array([
    [1.00,  0.10,  0.05],  # S&P 500
    [0.10,  1.00, -0.15],  # Treasury
    [0.05, -0.15,  1.00]   # Gold
])

# Portfolio scenarios
portfolios_gold = {
    'Traditional 60/40': np.array([0.60, 0.40, 0.00]),
    'With 10% Gold': np.array([0.50, 0.40, 0.10]),
    'With 15% Gold': np.array([0.45, 0.40, 0.15]),
    'With 20% Gold': np.array([0.40, 0.40, 0.20])
}

# Calculate covariance matrix
cov_matrix_gold = covariance_from_correlation(volatilities_gold, correlation_matrix_gold)

# Analyze each portfolio
print("="*90)
print("PORTFOLIO COMPARISON: IMPACT OF ADDING GOLD")
print("="*90)

results_gold = {}
for name, weights in portfolios_gold.items():
    # Expected return
    exp_ret = portfolio_expected_return(weights, expected_returns_gold)
    
    # Portfolio volatility
    port_vol = portfolio_volatility(weights, cov_matrix_gold)
    
    # Variance decomposition
    decomp = variance_decomposition(weights, cov_matrix_gold)
    
    # Sharpe ratio (assuming 2% risk-free rate)
    rf = 0.02
    sharpe = (exp_ret - rf) / port_vol
    
    results_gold[name] = {
        'weights': weights,
        'return': exp_ret,
        'volatility': port_vol,
        'sharpe': sharpe,
        'decomposition': decomp
    }
    
    print(f"\n{name}:")
    print(f"  Allocation: {weights[0]:.0%} Stocks, {weights[1]:.0%} Bonds, {weights[2]:.0%} Gold")
    print(f"  Expected Return:  {exp_ret*100:6.2f}%")
    print(f"  Volatility:       {port_vol*100:6.2f}%")
    print(f"  Sharpe Ratio:     {sharpe:6.2f}")
    
    # Print contribution of each asset to total variance
    total_var = decomp['total']
    print(f"  Variance Breakdown:")
    for i, asset in enumerate(asset_names_gold):
        contrib_pct = (decomp['individual'][i] / total_var * 100) if total_var > 0 else 0
        print(f"    - {asset:15s}: {decomp['individual'][i]:.5f} ({contrib_pct:5.1f}%)")
    interaction_pct = (decomp['interaction'] / total_var * 100) if total_var > 0 else 0
    print(f"    - Interaction:      {decomp['interaction']:.5f} ({interaction_pct:5.1f}%)")

# Comparison vs base case
base_case = results_gold['Traditional 60/40']
print("\n" + "="*90)
print("INCREMENTAL ANALYSIS vs TRADITIONAL 60/40")
print("="*90)

for name, result in results_gold.items():
    if name != 'Traditional 60/40':
        delta_ret = (result['return'] - base_case['return']) * 100
        delta_vol = (result['volatility'] - base_case['volatility']) * 100
        delta_sharpe = result['sharpe'] - base_case['sharpe']
        
        print(f"\n{name}:")
        print(f"  Return Change:     {delta_ret:+6.2f}% (from {base_case['return']*100:.2f}% to {result['return']*100:.2f}%)")
        print(f"  Volatility Change: {delta_vol:+6.2f}% (from {base_case['volatility']*100:.2f}% to {result['volatility']*100:.2f}%)")
        print(f"  Sharpe Change:     {delta_sharpe:+6.2f} (from {base_case['sharpe']:.2f} to {result['sharpe']:.2f})")

# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# Plot 1: Correlation Matrix Heatmap
ax = axes[0, 0]
sns.heatmap(correlation_matrix_gold, annot=True, fmt='.2f', cmap='RdYlGn', 
           center=0, vmin=-1, vmax=1, square=True, cbar_kws={'label': 'Correlation'},
           xticklabels=asset_names_gold, yticklabels=asset_names_gold, ax=ax)
ax.set_title('Asset Correlation Matrix\n(Gold provides diversification)', 
            fontsize=12, fontweight='bold')

# Plot 2: Risk-Return Scatter
ax = axes[0, 1]
for name, result in results_gold.items():
    ax.scatter(result['volatility']*100, result['return']*100, s=200, alpha=0.7,
              label=name, edgecolors='black', linewidth=2)
    ax.annotate(f"SR={result['sharpe']:.2f}", 
               xy=(result['volatility']*100, result['return']*100),
               xytext=(5, 5), textcoords='offset points', fontsize=8)

# Add individual assets
for i, asset in enumerate(asset_names_gold):
    ax.scatter(volatilities_gold[i]*100, expected_returns_gold[i]*100, 
              marker='D', s=150, alpha=0.5, edgecolors='red', linewidth=2)
    ax.annotate(asset, xy=(volatilities_gold[i]*100, expected_returns_gold[i]*100),
               xytext=(5, -10), textcoords='offset points', fontsize=9, 
               fontweight='bold', color='red')

ax.set_xlabel('Volatility (%)')
ax.set_ylabel('Expected Return (%)')
ax.set_title('Risk-Return Trade-off\n(Diamonds = Individual Assets)', 
            fontsize=12, fontweight='bold')
ax.legend(loc='best', fontsize=8)
ax.grid(True, alpha=0.3)

# Plot 3: Variance Decomposition Comparison
ax = axes[1, 0]
x_pos = np.arange(len(portfolios_gold))
width = 0.2

for i, asset in enumerate(asset_names_gold):
    contributions = [results_gold[name]['decomposition']['individual'][i] 
                    for name in portfolios_gold.keys()]
    ax.bar(x_pos + i*width, contributions, width, label=asset, alpha=0.8)

ax.set_xlabel('Portfolio')
ax.set_ylabel('Variance Contribution')
ax.set_title('Individual Asset Variance Contributions', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos + width)
ax.set_xticklabels([n.replace(' ', '\n') for n in portfolios_gold.keys()], fontsize=8)
ax.legend(loc='best')
ax.grid(True, alpha=0.3, axis='y')

# Plot 4: Sharpe Ratio Comparison
ax = axes[1, 1]
sharpe_ratios = [result['sharpe'] for result in results_gold.values()]
colors_sharpe = ['gray' if i == 0 else 'steelblue' for i in range(len(sharpe_ratios))]
bars = ax.bar(portfolios_gold.keys(), sharpe_ratios, color=colors_sharpe, 
             alpha=0.7, edgecolor='black', linewidth=2)

# Highlight best Sharpe ratio
best_idx = np.argmax(sharpe_ratios)
bars[best_idx].set_color('green')
bars[best_idx].set_alpha(0.9)

for i, (bar, sr) in enumerate(zip(bars, sharpe_ratios)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
           f'{sr:.3f}', ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Sharpe Ratio')
ax.set_title('Sharpe Ratio Comparison\n(Green = Best Risk-Adjusted Return)', 
            fontsize=12, fontweight='bold')
ax.set_xticklabels([n.replace(' ', '\n') for n in portfolios_gold.keys()], fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=base_case['sharpe'], color='red', linestyle='--', 
          linewidth=2, alpha=0.5, label=f"60/40 Base = {base_case['sharpe']:.3f}")
ax.legend(loc='best')

plt.tight_layout()
plt.show()

# Print covariance matrix for reference
print("\n" + "="*90)
print("COVARIANCE MATRIX")
print("="*90)
print("\nFull Covariance Matrix:")
cov_df = pd.DataFrame(cov_matrix_gold, index=asset_names_gold, columns=asset_names_gold)
print(cov_df.to_string(float_format=lambda x: f'{x:.6f}'))

print("\n" + "="*90)
print("KEY INSIGHTS")
print("="*90)
print("\n1. DIVERSIFICATION BENEFIT:")
print("   Adding Gold (with low/negative correlation) REDUCES portfolio volatility")
print("   despite Gold itself having HIGH volatility (20% vs 18% stocks, 7% bonds)")

best_portfolio = max(results_gold.items(), key=lambda x: x[1]['sharpe'])
print(f"\n2. OPTIMAL ALLOCATION:")
print(f"   {best_portfolio[0]} has the highest Sharpe ratio ({best_portfolio[1]['sharpe']:.3f})")
print(f"   Risk-adjusted returns IMPROVED by adding gold in moderation")

print("\n3. CORRELATION STRUCTURE:")
print("   - Stocks-Bonds:  ρ = +0.10 (mild diversification)")
print("   - Stocks-Gold:   ρ = +0.05 (strong diversification)")
print("   - Bonds-Gold:    ρ = -0.15 (negative correlation, best for hedging)")

print("\n4. VARIANCE DECOMPOSITION:")
print("   Interaction term is NEGATIVE due to low correlations")
print("   This negative interaction is the mathematical source of diversification benefit!")

print("\nConclusion: Gold's low correlation makes it valuable despite high individual volatility.")

## 4. Efficient Frontier - Two Asset Case

### Theory

The **efficient frontier** represents the set of portfolios that offer the highest expected return for each level of risk. For two assets, we can derive this analytically.

**Key Concepts:**

1. **Efficient vs Inefficient**: A portfolio is efficient if no other portfolio has higher return at same risk or lower risk at same return
2. **Hyperbolic Shape**: In mean-variance space, the two-asset frontier forms a hyperbola
3. **Minimum Variance Portfolio (MVP)**: The leftmost point on the efficient frontier (lowest risk)
4. **Correlation Dependency**: The shape of the frontier depends critically on correlation

**Mathematical Properties:**

For two assets with correlation ρ:

1. **ρ = +1 (Perfect Positive)**: Frontier is a straight line, no diversification
2. **ρ = -1 (Perfect Negative)**: Can create zero-variance portfolio
3. **-1 < ρ < 1**: Frontier curves (hyperbola), diversification benefit exists

**Real-World Examples:**

- **60/40 Portfolio**: Classic efficient allocation of stocks/bonds
- **Risk Parity**: Equal risk contribution from each asset
- **Minimum Variance**: Often heavily weighted toward low-volatility asset
- **Tactical Shifts**: Moving along frontier based on market outlook

**Why Two-Asset Case Matters:**

1. **Analytical Solution**: Can solve exactly without numerical optimization
2. **Intuition Building**: Understand mechanics before tackling N-asset case
3. **Asset Class Allocation**: Many decisions reduce to 2-asset choice (stocks vs bonds, domestic vs international)
4. **Overlay Strategies**: Adding a hedge (e.g., gold, puts) to existing portfolio

**The Frontier Equation:**

For weight w in asset 1 and (1-w) in asset 2:

$$\sigma_p(w) = \sqrt{w^2\sigma_1^2 + (1-w)^2\sigma_2^2 + 2w(1-w)\rho\sigma_1\sigma_2}$$

$$\mu_p(w) = w\mu_1 + (1-w)\mu_2$$

Plotting $\mu_p$ vs $\sigma_p$ as w varies from 0 to 1 gives the efficient frontier.

### Mathematical Formulation

For a two-asset portfolio, we can derive the **minimum variance portfolio** analytically.

**Portfolio Return and Variance:**

$$
\begin{aligned}
\mu_p &= w_1\mu_1 + w_2\mu_2 = w_1\mu_1 + (1-w_1)\mu_2 \\
\sigma_p^2 &= w_1^2\sigma_1^2 + w_2^2\sigma_2^2 + 2w_1w_2\rho_{12}\sigma_1\sigma_2
\end{aligned}
$$

**Minimum Variance Portfolio Weight:**

To find the weight $w_1^{MVP}$ that minimizes portfolio variance, we differentiate $\sigma_p^2$ with respect to $w_1$ and set equal to zero:

$$
\frac{\partial \sigma_p^2}{\partial w_1} = 2w_1\sigma_1^2 - 2(1-w_1)\sigma_2^2 + 2(1-2w_1)\rho_{12}\sigma_1\sigma_2 = 0
$$

Solving for $w_1$:

$$
w_1^{MVP} = \frac{\sigma_2^2 - \rho_{12}\sigma_1\sigma_2}{\sigma_1^2 + \sigma_2^2 - 2\rho_{12}\sigma_1\sigma_2}
$$

And therefore: $w_2^{MVP} = 1 - w_1^{MVP}$

**Special Cases:**

1. **Uncorrelated Assets** ($\rho_{12} = 0$):
   $$w_1^{MVP} = \frac{\sigma_2^2}{\sigma_1^2 + \sigma_2^2}$$
   
   The MVP weights are inversely proportional to variances!

2. **Equal Volatility** ($\sigma_1 = \sigma_2 = \sigma$):
   $$w_1^{MVP} = \frac{1 - \rho_{12}}{2(1 - \rho_{12})} = 0.5$$
   
   Equal split when assets have equal volatility!

3. **Perfect Negative Correlation** ($\rho_{12} = -1$):
   $$w_1^{MVP} = \frac{\sigma_2}{\sigma_1 + \sigma_2}$$
   
   Can achieve zero variance!

**Parametric Frontier:**

As $w_1$ varies from 0 to 1:

$$
\begin{cases}
\sigma_p(w_1) = \sqrt{w_1^2\sigma_1^2 + (1-w_1)^2\sigma_2^2 + 2w_1(1-w_1)\rho_{12}\sigma_1\sigma_2} \\
\mu_p(w_1) = w_1\mu_1 + (1-w_1)\mu_2
\end{cases}
$$

This traces out a curve in $(\sigma_p, \mu_p)$ space - the **efficient frontier**.

In [ ]:
# Python implementation for Two-Asset Efficient Frontier

def minimum_variance_portfolio_2asset(sigma1, sigma2, rho):
    """
    Calculate minimum variance portfolio weights for two assets.
    
    Parameters
    ----------
    sigma1 : float
        Volatility of asset 1
    sigma2 : float
        Volatility of asset 2
    rho : float
        Correlation between assets
    
    Returns
    -------
    np.ndarray
        Weights [w1, w2] for minimum variance portfolio
    
    Notes
    -----
    Analytical formula:
        w1 = (sigma2^2 - rho*sigma1*sigma2) / (sigma1^2 + sigma2^2 - 2*rho*sigma1*sigma2)
        w2 = 1 - w1
    
    Examples
    --------
    >>> minimum_variance_portfolio_2asset(0.20, 0.08, 0.3)
    array([0.135, 0.865])
    """
    numerator = sigma2**2 - rho * sigma1 * sigma2
    denominator = sigma1**2 + sigma2**2 - 2 * rho * sigma1 * sigma2
    
    # Handle edge case where denominator is zero (perfect correlation)
    if abs(denominator) < 1e-10:
        w1 = 0.5
    else:
        w1 = numerator / denominator
    
    w2 = 1 - w1
    
    return np.array([w1, w2])


def efficient_frontier_2asset(mu1, mu2, sigma1, sigma2, rho, num_points=100, 
                               allow_short=False):
    """
    Generate efficient frontier for two-asset portfolio.
    
    Parameters
    ----------
    mu1 : float
        Expected return of asset 1
    mu2 : float
        Expected return of asset 2
    sigma1 : float
        Volatility of asset 1
    sigma2 : float
        Volatility of asset 2
    rho : float
        Correlation between assets
    num_points : int, optional
        Number of points on frontier (default: 100)
    allow_short : bool, optional
        If True, allow short selling (w < 0 or w > 1), default False
    
    Returns
    -------
    dict
        Dictionary containing:
        - 'weights': Array of weight pairs, shape (num_points, 2)
        - 'returns': Array of portfolio returns
        - 'volatilities': Array of portfolio volatilities
        - 'sharpe_ratios': Array of Sharpe ratios (assuming rf=0)
    
    Notes
    -----
    Generates frontier by varying weight from 0 to 1 (or beyond if allow_short=True)
    
    Examples
    --------
    >>> frontier = efficient_frontier_2asset(0.10, 0.04, 0.18, 0.07, 0.3)
    >>> plt.plot(frontier['volatilities'], frontier['returns'])
    """
    # Weight range
    if allow_short:
        w1_range = np.linspace(-0.5, 1.5, num_points)
    else:
        w1_range = np.linspace(0, 1, num_points)
    
    weights = np.zeros((num_points, 2))
    returns = np.zeros(num_points)
    volatilities = np.zeros(num_points)
    
    # Covariance matrix
    corr_matrix = np.array([[1, rho], [rho, 1]])
    cov_matrix = covariance_from_correlation(np.array([sigma1, sigma2]), corr_matrix)
    mu = np.array([mu1, mu2])
    
    for i, w1 in enumerate(w1_range):
        w = np.array([w1, 1 - w1])
        weights[i] = w
        returns[i] = portfolio_expected_return(w, mu)
        volatilities[i] = portfolio_volatility(w, cov_matrix)
    
    # Calculate Sharpe ratios (assuming rf = 0 for simplicity)
    sharpe_ratios = returns / volatilities
    
    return {
        'weights': weights,
        'returns': returns,
        'volatilities': volatilities,
        'sharpe_ratios': sharpe_ratios
    }


def find_max_sharpe_2asset(mu1, mu2, sigma1, sigma2, rho, rf=0.02):
    """
    Find maximum Sharpe ratio portfolio for two assets.
    
    Parameters
    ----------
    mu1 : float
        Expected return of asset 1
    mu2 : float
        Expected return of asset 2
    sigma1 : float
        Volatility of asset 1
    sigma2 : float
        Volatility of asset 2
    rho : float
        Correlation between assets
    rf : float, optional
        Risk-free rate (default: 0.02)
    
    Returns
    -------
    dict
        Dictionary containing:
        - 'weights': Optimal weights [w1, w2]
        - 'return': Portfolio return
        - 'volatility': Portfolio volatility
        - 'sharpe': Sharpe ratio
    
    Notes
    -----
    Uses numerical optimization to find maximum Sharpe ratio.
    For two assets, this is the tangency portfolio.
    
    Examples
    --------
    >>> result = find_max_sharpe_2asset(0.10, 0.04, 0.18, 0.07, 0.3, rf=0.02)
    >>> print(result['weights'])
    """
    # Objective: minimize negative Sharpe ratio
    def neg_sharpe(w1):
        w = np.array([w1, 1 - w1])
        corr_matrix = np.array([[1, rho], [rho, 1]])
        cov_matrix = covariance_from_correlation(np.array([sigma1, sigma2]), corr_matrix)
        mu = np.array([mu1, mu2])
        
        port_return = portfolio_expected_return(w, mu)
        port_vol = portfolio_volatility(w, cov_matrix)
        
        if port_vol == 0:
            return 1e10
        
        sharpe = (port_return - rf) / port_vol
        return -sharpe
    
    # Optimize (long-only constraint)
    from scipy.optimize import minimize_scalar
    result = minimize_scalar(neg_sharpe, bounds=(0, 1), method='bounded')
    
    optimal_w1 = result.x
    optimal_weights = np.array([optimal_w1, 1 - optimal_w1])
    
    # Calculate portfolio statistics
    corr_matrix = np.array([[1, rho], [rho, 1]])
    cov_matrix = covariance_from_correlation(np.array([sigma1, sigma2]), corr_matrix)
    mu = np.array([mu1, mu2])
    
    optimal_return = portfolio_expected_return(optimal_weights, mu)
    optimal_vol = portfolio_volatility(optimal_weights, cov_matrix)
    optimal_sharpe = (optimal_return - rf) / optimal_vol
    
    return {
        'weights': optimal_weights,
        'return': optimal_return,
        'volatility': optimal_vol,
        'sharpe': optimal_sharpe
    }


print("[OK] Two-asset efficient frontier functions implemented")
print("\nFunction capabilities:")
print("  - minimum_variance_portfolio_2asset(): Analytical MVP weights")
print("  - efficient_frontier_2asset(): Generate complete frontier")
print("  - find_max_sharpe_2asset(): Find tangency portfolio (max Sharpe)")

In [ ]:
# Visualization for Two-Asset Efficient Frontier

# Define asset parameters (Stocks vs Bonds)
mu1, mu2 = 0.10, 0.04      # Expected returns: 10% stocks, 4% bonds
sigma1, sigma2 = 0.18, 0.07  # Volatilities: 18% stocks, 7% bonds
asset_names_ef = ['Stocks', 'Bonds']

# Test different correlations
correlations_ef = [-0.5, 0.0, 0.3, 0.9]
colors_ef = ['green', 'blue', 'orange', 'red']

# Create 2x2 subplot
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: Efficient Frontier for Different Correlations
ax = axes[0, 0]

for rho, color in zip(correlations_ef, colors_ef):
    frontier = efficient_frontier_2asset(mu1, mu2, sigma1, sigma2, rho, num_points=200)
    ax.plot(frontier['volatilities'] * 100, frontier['returns'] * 100, 
           linewidth=2.5, label=f'ρ = {rho:+.1f}', color=color, alpha=0.8)
    
    # Mark minimum variance portfolio
    mvp_weights = minimum_variance_portfolio_2asset(sigma1, sigma2, rho)
    corr_mat = np.array([[1, rho], [rho, 1]])
    cov_mat = covariance_from_correlation(np.array([sigma1, sigma2]), corr_mat)
    mvp_vol = portfolio_volatility(mvp_weights, cov_mat)
    mvp_ret = portfolio_expected_return(mvp_weights, np.array([mu1, mu2]))
    ax.plot(mvp_vol * 100, mvp_ret * 100, 'o', markersize=8, color=color)

# Mark individual assets
ax.plot(sigma1 * 100, mu1 * 100, 's', markersize=15, color='black', 
       label='Stocks', zorder=5)
ax.plot(sigma2 * 100, mu2 * 100, 's', markersize=15, color='gray', 
       label='Bonds', zorder=5)

ax.set_xlabel('Portfolio Volatility (%)', fontsize=11)
ax.set_ylabel('Expected Return (%)', fontsize=11)
ax.set_title('Efficient Frontier: Impact of Correlation\n(Circles = Minimum Variance Portfolios)', 
            fontsize=12, fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)

# Plot 2: Optimal Weights Path Along Frontier
ax = axes[0, 1]

# Use ρ = 0.3 (realistic stocks-bonds correlation)
rho_default = 0.3
frontier_default = efficient_frontier_2asset(mu1, mu2, sigma1, sigma2, rho_default, num_points=200)

# Plot weights vs volatility
ax.plot(frontier_default['volatilities'] * 100, 
       frontier_default['weights'][:, 0] * 100,
       linewidth=2.5, color='steelblue', label='Stock Weight')
ax.plot(frontier_default['volatilities'] * 100, 
       frontier_default['weights'][:, 1] * 100,
       linewidth=2.5, color='coral', label='Bond Weight')

# Mark special portfolios
mvp_weights_def = minimum_variance_portfolio_2asset(sigma1, sigma2, rho_default)
corr_mat_def = np.array([[1, rho_default], [rho_default, 1]])
cov_mat_def = covariance_from_correlation(np.array([sigma1, sigma2]), corr_mat_def)
mvp_vol_def = portfolio_volatility(mvp_weights_def, cov_mat_def)

ax.axvline(x=mvp_vol_def * 100, color='red', linestyle='--', linewidth=2, 
          alpha=0.5, label=f'Min Vol = {mvp_vol_def*100:.2f}%')
ax.axhline(y=50, color='gray', linestyle=':', alpha=0.5)

# Mark 60/40 portfolio
w_6040 = np.array([0.6, 0.4])
vol_6040 = portfolio_volatility(w_6040, cov_mat_def)
ax.plot(vol_6040 * 100, 60, 'ro', markersize=12, label='60/40 Portfolio', zorder=5)

ax.set_xlabel('Portfolio Volatility (%)', fontsize=11)
ax.set_ylabel('Asset Weight (%)', fontsize=11)
ax.set_title(f'Portfolio Weights Along Frontier\n(ρ = {rho_default:+.1f})', 
            fontsize=12, fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 100)

# Plot 3: Sharpe Ratio Curve
ax = axes[1, 0]

# Calculate Sharpe ratios for different correlations
rf = 0.02

for rho, color in zip(correlations_ef, colors_ef):
    frontier = efficient_frontier_2asset(mu1, mu2, sigma1, sigma2, rho, num_points=200)
    sharpe_ratios = (frontier['returns'] - rf) / frontier['volatilities']
    
    ax.plot(frontier['weights'][:, 0] * 100, sharpe_ratios, 
           linewidth=2.5, label=f'ρ = {rho:+.1f}', color=color, alpha=0.8)
    
    # Mark maximum Sharpe ratio
    max_sharpe_result = find_max_sharpe_2asset(mu1, mu2, sigma1, sigma2, rho, rf)
    ax.plot(max_sharpe_result['weights'][0] * 100, max_sharpe_result['sharpe'], 
           'o', markersize=8, color=color)
    
    # Annotate max Sharpe
    ax.annotate(f'{max_sharpe_result["sharpe"]:.2f}', 
               xy=(max_sharpe_result['weights'][0] * 100, max_sharpe_result['sharpe']),
               xytext=(5, 5), textcoords='offset points', fontsize=8)

ax.set_xlabel('Stock Weight (%)', fontsize=11)
ax.set_ylabel('Sharpe Ratio', fontsize=11)
ax.set_title(f'Sharpe Ratio vs Stock Allocation\n(rf = {rf:.1%}, Circles = Maximum)', 
            fontsize=12, fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

# Plot 4: Risk Decomposition Along Frontier
ax = axes[1, 1]

# Use ρ = 0.3
frontier_risk = efficient_frontier_2asset(mu1, mu2, sigma1, sigma2, rho_default, num_points=100)

# Calculate variance components
variances_total = frontier_risk['volatilities']**2
variances_stock = (frontier_risk['weights'][:, 0]**2) * sigma1**2
variances_bond = (frontier_risk['weights'][:, 1]**2) * sigma2**2
variances_interaction = variances_total - variances_stock - variances_bond

# Stack plot
ax.fill_between(frontier_risk['weights'][:, 0] * 100, 0, variances_stock, 
               alpha=0.6, label='Stock Variance', color='steelblue')
ax.fill_between(frontier_risk['weights'][:, 0] * 100, variances_stock, 
               variances_stock + variances_bond,
               alpha=0.6, label='Bond Variance', color='coral')
ax.fill_between(frontier_risk['weights'][:, 0] * 100, 
               variances_stock + variances_bond,
               variances_total,
               alpha=0.6, label='Interaction (Covariance)', color='lightgreen')

# Plot total variance line
ax.plot(frontier_risk['weights'][:, 0] * 100, variances_total, 
       'k-', linewidth=2.5, label='Total Variance')

# Mark MVP
ax.axvline(x=mvp_weights_def[0] * 100, color='red', linestyle='--', 
          linewidth=2, alpha=0.7, label=f'Min Variance ({mvp_weights_def[0]*100:.1f}% stocks)')

ax.set_xlabel('Stock Weight (%)', fontsize=11)
ax.set_ylabel('Variance', fontsize=11)
ax.set_title(f'Variance Decomposition Along Frontier\n(ρ = {rho_default:+.1f})', 
            fontsize=12, fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print detailed analysis
print("="*90)
print("TWO-ASSET EFFICIENT FRONTIER ANALYSIS")
print("="*90)
print(f"\nAsset Parameters:")
print(f"  Stocks: μ = {mu1:.1%}, σ = {sigma1:.1%}")
print(f"  Bonds:  μ = {mu2:.1%}, σ = {sigma2:.1%}")

print("\n" + "-"*90)
print("MINIMUM VARIANCE PORTFOLIOS (Different Correlations)")
print("-"*90)

for rho in correlations_ef:
    mvp_w = minimum_variance_portfolio_2asset(sigma1, sigma2, rho)
    corr_mat = np.array([[1, rho], [rho, 1]])
    cov_mat = covariance_from_correlation(np.array([sigma1, sigma2]), corr_mat)
    mvp_vol = portfolio_volatility(mvp_w, cov_mat)
    mvp_ret = portfolio_expected_return(mvp_w, np.array([mu1, mu2]))
    
    print(f"\nCorrelation ρ = {rho:+.1f}:")
    print(f"  Optimal Weights: {mvp_w[0]*100:.1f}% Stocks, {mvp_w[1]*100:.1f}% Bonds")
    print(f"  Portfolio Vol:   {mvp_vol*100:.2f}%")
    print(f"  Expected Return: {mvp_ret*100:.2f}%")
    print(f"  Formula Check:   w_stocks = σ²_bonds - ρσ_stocks σ_bonds")
    print(f"                            ─────────────────────────────────")
    print(f"                            σ²_stocks + σ²_bonds - 2ρσ_stocks σ_bonds")
    
print("\n" + "-"*90)
print("MAXIMUM SHARPE RATIO PORTFOLIOS (Tangency Portfolios)")
print("-"*90)

for rho in correlations_ef:
    max_sharpe_res = find_max_sharpe_2asset(mu1, mu2, sigma1, sigma2, rho, rf=0.02)
    
    print(f"\nCorrelation ρ = {rho:+.1f}:")
    print(f"  Optimal Weights: {max_sharpe_res['weights'][0]*100:.1f}% Stocks, "
          f"{max_sharpe_res['weights'][1]*100:.1f}% Bonds")
    print(f"  Expected Return: {max_sharpe_res['return']*100:.2f}%")
    print(f"  Volatility:      {max_sharpe_res['volatility']*100:.2f}%")
    print(f"  Sharpe Ratio:    {max_sharpe_res['sharpe']:.3f}")

print("\n" + "="*90)
print("KEY INSIGHTS")
print("="*90)
print("\n1. CORRELATION IMPACT:")
print("   - Lower correlation → More curved frontier → Greater diversification")
print("   - At ρ = -0.5: Can achieve very low volatility portfolios")
print("   - At ρ = +0.9: Frontier nearly straight, limited diversification")

print("\n2. MINIMUM VARIANCE PORTFOLIO:")
print("   - Moves toward lower-volatility asset as correlation increases")
print("   - For uncorrelated assets: weights ∝ 1/σ² (inverse variance weighting)")
print("   - Always feasible (weights sum to 1)")

print("\n3. MAXIMUM SHARPE RATIO:")
print("   - Balances return AND risk, not just minimizing risk")
print("   - Typically more aggressive (higher stock weight) than MVP")
print("   - This is the 'tangency portfolio' in classical MPT")

print("\n4. RISK DECOMPOSITION:")
print("   - Interaction term can be negative (diversification benefit)")
print("   - At MVP, marginal contribution of both assets is equal")
print("   - Moving away from MVP increases variance quadratically")

### Practical Example

Let's solve a real portfolio management problem: An investor wants to allocate between US Stocks (S&P 500) and International Stocks (MSCI EAFE).

**Scenario:**
- US Stocks (S&P 500): Expected Return = 9%, Volatility = 16%
- International Stocks (EAFE): Expected Return = 8%, Volatility = 18%
- Historical Correlation = 0.75 (high but not perfect)
- Risk-free rate = 2.5%

**Questions:**
1. What is the minimum variance allocation?
2. What allocation maximizes the Sharpe ratio?
3. How does the efficient frontier change if correlation drops to 0.5 (post-diversification)?
4. What is the optimal allocation for an investor with 12% volatility target?

In [ ]:
# Practical example: US vs International Stocks Allocation

# Define parameters
mu_us, mu_intl = 0.09, 0.08        # Expected returns
sigma_us, sigma_intl = 0.16, 0.18  # Volatilities
rho_current = 0.75                  # Current correlation
rho_future = 0.50                   # Future (lower) correlation scenario
rf_rate = 0.025                     # Risk-free rate
vol_target = 0.12                   # Target volatility (12%)

asset_names_practical = ['US Stocks (S&P 500)', 'International (EAFE)']

print("="*90)
print("PRACTICAL PORTFOLIO ALLOCATION PROBLEM")
print("="*90)
print("\nScenario: Allocating between US and International Stocks")
print(f"\nAsset Parameters:")
print(f"  {asset_names_practical[0]:25s}: μ = {mu_us:.1%}, σ = {sigma_us:.1%}")
print(f"  {asset_names_practical[1]:25s}: μ = {mu_intl:.1%}, σ = {sigma_intl:.1%}")
print(f"\nMarket Conditions:")
print(f"  Current Correlation:  ρ = {rho_current:.2f}")
print(f"  Risk-Free Rate:       rf = {rf_rate:.2%}")

# Question 1: Minimum Variance Portfolio
print("\n" + "="*90)
print("QUESTION 1: MINIMUM VARIANCE ALLOCATION")
print("="*90)

mvp_current = minimum_variance_portfolio_2asset(sigma_us, sigma_intl, rho_current)
corr_mat_curr = np.array([[1, rho_current], [rho_current, 1]])
cov_mat_curr = covariance_from_correlation(np.array([sigma_us, sigma_intl]), corr_mat_curr)
mvp_vol_curr = portfolio_volatility(mvp_current, cov_mat_curr)
mvp_ret_curr = portfolio_expected_return(mvp_current, np.array([mu_us, mu_intl]))

print(f"\nMinimum Variance Portfolio (ρ = {rho_current}):")
print(f"  US Stocks:        {mvp_current[0]*100:6.2f}%")
print(f"  International:    {mvp_current[1]*100:6.2f}%")
print(f"  Portfolio Vol:    {mvp_vol_curr*100:6.2f}%")
print(f"  Expected Return:  {mvp_ret_curr*100:6.2f}%")
print(f"  Sharpe Ratio:     {(mvp_ret_curr - rf_rate)/mvp_vol_curr:6.3f}")

print("\nInterpretation:")
print(f"  Despite international stocks having HIGHER volatility ({sigma_intl:.1%} vs {sigma_us:.1%}),")
print(f"  MVP allocates {mvp_current[1]*100:.1f}% to international due to diversification benefit.")

# Question 2: Maximum Sharpe Ratio Portfolio
print("\n" + "="*90)
print("QUESTION 2: MAXIMUM SHARPE RATIO ALLOCATION (TANGENCY PORTFOLIO)")
print("="*90)

max_sharpe_current = find_max_sharpe_2asset(mu_us, mu_intl, sigma_us, sigma_intl, 
                                             rho_current, rf=rf_rate)

print(f"\nTangency Portfolio (ρ = {rho_current}):")
print(f"  US Stocks:        {max_sharpe_current['weights'][0]*100:6.2f}%")
print(f"  International:    {max_sharpe_current['weights'][1]*100:6.2f}%")
print(f"  Expected Return:  {max_sharpe_current['return']*100:6.2f}%")
print(f"  Volatility:       {max_sharpe_current['volatility']*100:6.2f}%")
print(f"  Sharpe Ratio:     {max_sharpe_current['sharpe']:6.3f}")

print("\nComparison with MVP:")
print(f"  Sharpe Ratio:     {max_sharpe_current['sharpe']:.3f} vs {(mvp_ret_curr - rf_rate)/mvp_vol_curr:.3f}")
print(f"  US Allocation:    {max_sharpe_current['weights'][0]*100:.1f}% vs {mvp_current[0]*100:.1f}%")
print(f"\nInterpretation:")
print(f"  Max Sharpe tilts MORE toward US stocks (higher expected return)")
print(f"  Accepts higher volatility to achieve better risk-adjusted returns")

# Question 3: Impact of Lower Correlation
print("\n" + "="*90)
print("QUESTION 3: SCENARIO ANALYSIS - CORRELATION DROPS TO 0.50")
print("="*90)

mvp_future = minimum_variance_portfolio_2asset(sigma_us, sigma_intl, rho_future)
corr_mat_fut = np.array([[1, rho_future], [rho_future, 1]])
cov_mat_fut = covariance_from_correlation(np.array([sigma_us, sigma_intl]), corr_mat_fut)
mvp_vol_fut = portfolio_volatility(mvp_future, cov_mat_fut)
mvp_ret_fut = portfolio_expected_return(mvp_future, np.array([mu_us, mu_intl]))

max_sharpe_future = find_max_sharpe_2asset(mu_us, mu_intl, sigma_us, sigma_intl, 
                                            rho_future, rf=rf_rate)

print(f"\nImpact on Minimum Variance Portfolio:")
print(f"  Current (ρ={rho_current}):  {mvp_current[0]*100:5.1f}% US, Vol = {mvp_vol_curr*100:5.2f}%")
print(f"  Future (ρ={rho_future}):   {mvp_future[0]*100:5.1f}% US, Vol = {mvp_vol_fut*100:5.2f}%")
print(f"  Change:           {(mvp_future[0] - mvp_current[0])*100:+5.1f}% US allocation")
print(f"  Vol Reduction:    {(mvp_vol_fut - mvp_vol_curr)*100:+5.2f}%")

print(f"\nImpact on Maximum Sharpe Portfolio:")
print(f"  Current (ρ={rho_current}):  SR = {max_sharpe_current['sharpe']:.3f}")
print(f"  Future (ρ={rho_future}):   SR = {max_sharpe_future['sharpe']:.3f}")
print(f"  Improvement:      {(max_sharpe_future['sharpe'] - max_sharpe_current['sharpe']):+.3f}")

print("\nInterpretation:")
print("  Lower correlation IMPROVES diversification:")
print("  - Minimum variance portfolio achieves LOWER risk")
print("  - Maximum Sharpe ratio INCREASES (better risk-adjusted returns)")
print("  - More balanced allocation between US and international")

# Question 4: Target Volatility Portfolio
print("\n" + "="*90)
print(f"QUESTION 4: OPTIMAL ALLOCATION FOR {vol_target*100:.0f}% VOLATILITY TARGET")
print("="*90)

# Find weight that achieves target volatility
def find_target_vol_weight(target_vol, sigma1, sigma2, rho):
    """Find weight that achieves target volatility."""
    def vol_diff(w1):
        w = np.array([w1, 1 - w1])
        corr_mat = np.array([[1, rho], [rho, 1]])
        cov_mat = covariance_from_correlation(np.array([sigma1, sigma2]), corr_mat)
        return (portfolio_volatility(w, cov_mat) - target_vol)**2
    
    from scipy.optimize import minimize_scalar
    result = minimize_scalar(vol_diff, bounds=(0, 1), method='bounded')
    return result.x

# Current correlation scenario
w_us_target = find_target_vol_weight(vol_target, sigma_us, sigma_intl, rho_current)
w_target = np.array([w_us_target, 1 - w_us_target])
ret_target = portfolio_expected_return(w_target, np.array([mu_us, mu_intl]))
vol_target_actual = portfolio_volatility(w_target, cov_mat_curr)
sharpe_target = (ret_target - rf_rate) / vol_target_actual

print(f"\nTarget Volatility Portfolio (ρ = {rho_current}):")
print(f"  US Stocks:        {w_target[0]*100:6.2f}%")
print(f"  International:    {w_target[1]*100:6.2f}%")
print(f"  Expected Return:  {ret_target*100:6.2f}%")
print(f"  Actual Volatility:{vol_target_actual*100:6.2f}% (target: {vol_target*100:.2f}%)")
print(f"  Sharpe Ratio:     {sharpe_target:6.3f}")

print(f"\nComparison with Other Strategies:")
print(f"  {'Strategy':<25s} {'US%':>8s} {'Intl%':>8s} {'Return':>8s} {'Vol':>8s} {'Sharpe':>8s}")
print(f"  {'-'*25} {'-'*8} {'-'*8} {'-'*8} {'-'*8} {'-'*8}")
print(f"  {'100% US Stocks':<25s} {100:8.1f} {0:8.1f} {mu_us*100:8.2f} {sigma_us*100:8.2f} "
      f"{(mu_us - rf_rate)/sigma_us:8.3f}")
print(f"  {'100% International':<25s} {0:8.1f} {100:8.1f} {mu_intl*100:8.2f} {sigma_intl*100:8.2f} "
      f"{(mu_intl - rf_rate)/sigma_intl:8.3f}")
print(f"  {'Minimum Variance':<25s} {mvp_current[0]*100:8.1f} {mvp_current[1]*100:8.1f} "
      f"{mvp_ret_curr*100:8.2f} {mvp_vol_curr*100:8.2f} {(mvp_ret_curr - rf_rate)/mvp_vol_curr:8.3f}")
print(f"  {'Maximum Sharpe':<25s} {max_sharpe_current['weights'][0]*100:8.1f} "
      f"{max_sharpe_current['weights'][1]*100:8.1f} {max_sharpe_current['return']*100:8.2f} "
      f"{max_sharpe_current['volatility']*100:8.2f} {max_sharpe_current['sharpe']:8.3f}")
print(f"  {f'{vol_target*100:.0f}% Vol Target':<25s} {w_target[0]*100:8.1f} {w_target[1]*100:8.1f} "
      f"{ret_target*100:8.2f} {vol_target_actual*100:8.2f} {sharpe_target:8.3f}")

# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# Plot 1: Efficient Frontiers - Current vs Future Correlation
ax = axes[0, 0]

for rho, label, color in [(rho_current, f'Current ρ={rho_current}', 'blue'),
                           (rho_future, f'Future ρ={rho_future}', 'green')]:
    frontier = efficient_frontier_2asset(mu_us, mu_intl, sigma_us, sigma_intl, rho, num_points=200)
    ax.plot(frontier['volatilities'] * 100, frontier['returns'] * 100, 
           linewidth=2.5, label=label, alpha=0.8, color=color)

# Mark special portfolios for current scenario
ax.plot(mvp_vol_curr * 100, mvp_ret_curr * 100, 'o', markersize=12, 
       color='red', label='Min Variance (current)', zorder=5)
ax.plot(max_sharpe_current['volatility'] * 100, max_sharpe_current['return'] * 100, 
       '^', markersize=12, color='orange', label='Max Sharpe (current)', zorder=5)
ax.plot(vol_target_actual * 100, ret_target * 100, 's', markersize=12, 
       color='purple', label=f'{vol_target*100:.0f}% Vol Target', zorder=5)

# Mark individual assets
ax.plot(sigma_us * 100, mu_us * 100, 'D', markersize=10, color='black', 
       label='US Stocks', alpha=0.7)
ax.plot(sigma_intl * 100, mu_intl * 100, 'D', markersize=10, color='gray', 
       label='International', alpha=0.7)

# Capital Allocation Line (from rf through max Sharpe)
x_cal = np.array([0, max_sharpe_current['volatility'] * 1.5])
y_cal = rf_rate + max_sharpe_current['sharpe'] * x_cal
ax.plot(x_cal * 100, y_cal * 100, '--', linewidth=2, color='orange', 
       alpha=0.5, label='Capital Allocation Line')

ax.set_xlabel('Volatility (%)', fontsize=11)
ax.set_ylabel('Expected Return (%)', fontsize=11)
ax.set_title('Efficient Frontier: Current vs Future Correlation\n(Lower correlation shifts frontier left)', 
            fontsize=12, fontweight='bold')
ax.legend(loc='best', fontsize=8)
ax.grid(True, alpha=0.3)

# Plot 2: Allocation Comparison
ax = axes[0, 1]

strategies = ['100% US', '100% Intl', 'Min Var', 'Max Sharpe', f'{vol_target*100:.0f}% Vol']
us_weights = [100, 0, mvp_current[0]*100, max_sharpe_current['weights'][0]*100, w_target[0]*100]
intl_weights = [0, 100, mvp_current[1]*100, max_sharpe_current['weights'][1]*100, w_target[1]*100]

x_pos = np.arange(len(strategies))
width = 0.6

p1 = ax.bar(x_pos, us_weights, width, label='US Stocks', color='steelblue', alpha=0.8)
p2 = ax.bar(x_pos, intl_weights, width, bottom=us_weights, label='International', 
           color='coral', alpha=0.8)

# Add percentage labels
for i, (us, intl) in enumerate(zip(us_weights, intl_weights)):
    if us > 5:
        ax.text(i, us/2, f'{us:.0f}%', ha='center', va='center', 
               color='white', fontweight='bold', fontsize=9)
    if intl > 5:
        ax.text(i, us + intl/2, f'{intl:.0f}%', ha='center', va='center', 
               color='white', fontweight='bold', fontsize=9)

ax.set_ylabel('Portfolio Weight (%)', fontsize=11)
ax.set_title('Portfolio Allocation Comparison', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(strategies, rotation=15, ha='right')
ax.legend(loc='best')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 105)

# Plot 3: Sharpe Ratio Comparison
ax = axes[1, 0]

sharpe_ratios_comp = [
    (mu_us - rf_rate) / sigma_us,
    (mu_intl - rf_rate) / sigma_intl,
    (mvp_ret_curr - rf_rate) / mvp_vol_curr,
    max_sharpe_current['sharpe'],
    sharpe_target
]

colors_sharpe_comp = ['gray', 'gray', 'lightblue', 'green', 'purple']
bars = ax.bar(strategies, sharpe_ratios_comp, color=colors_sharpe_comp, 
             alpha=0.7, edgecolor='black', linewidth=2)

# Highlight best
best_idx = np.argmax(sharpe_ratios_comp)
bars[best_idx].set_edgecolor('gold')
bars[best_idx].set_linewidth(3)

for bar, sr in zip(bars, sharpe_ratios_comp):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
           f'{sr:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=9)

ax.set_ylabel('Sharpe Ratio', fontsize=11)
ax.set_title(f'Sharpe Ratio Comparison (rf = {rf_rate:.2%})\n(Gold border = Best)', 
            fontsize=12, fontweight='bold')
ax.set_xticklabels(strategies, rotation=15, ha='right')
ax.grid(True, alpha=0.3, axis='y')

# Plot 4: Sensitivity Analysis - Allocation vs Correlation
ax = axes[1, 1]

rho_range = np.linspace(0, 1, 50)
mvp_allocations = []
sharpe_allocations = []

for rho in rho_range:
    mvp_w = minimum_variance_portfolio_2asset(sigma_us, sigma_intl, rho)
    mvp_allocations.append(mvp_w[0] * 100)
    
    max_sr = find_max_sharpe_2asset(mu_us, mu_intl, sigma_us, sigma_intl, rho, rf=rf_rate)
    sharpe_allocations.append(max_sr['weights'][0] * 100)

ax.plot(rho_range, mvp_allocations, linewidth=2.5, label='Minimum Variance Portfolio', 
       color='blue')
ax.plot(rho_range, sharpe_allocations, linewidth=2.5, label='Maximum Sharpe Portfolio', 
       color='green')

# Mark current and future scenarios
ax.axvline(x=rho_current, color='red', linestyle='--', alpha=0.5, 
          label=f'Current ρ={rho_current}')
ax.axvline(x=rho_future, color='orange', linestyle='--', alpha=0.5, 
          label=f'Future ρ={rho_future}')

ax.set_xlabel('Correlation (ρ)', fontsize=11)
ax.set_ylabel('US Stocks Allocation (%)', fontsize=11)
ax.set_title('Optimal Allocation Sensitivity to Correlation', fontsize=12, fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

# Final recommendations
print("\n" + "="*90)
print("RECOMMENDATIONS")
print("="*90)

print("\n1. FOR RISK-AVERSE INVESTORS (Minimum Risk):")
print(f"   Allocate {mvp_current[0]*100:.0f}% US / {mvp_current[1]*100:.0f}% International")
print(f"   Expected: {mvp_ret_curr*100:.1f}% return, {mvp_vol_curr*100:.1f}% volatility")

print("\n2. FOR RISK-ADJUSTED RETURN MAXIMIZERS:")
print(f"   Allocate {max_sharpe_current['weights'][0]*100:.0f}% US / "
      f"{max_sharpe_current['weights'][1]*100:.0f}% International")
print(f"   Expected: {max_sharpe_current['return']*100:.1f}% return, "
      f"{max_sharpe_current['volatility']*100:.1f}% volatility")
print(f"   Sharpe Ratio: {max_sharpe_current['sharpe']:.3f} (best risk-adjusted returns)")

print(f"\n3. FOR {vol_target*100:.0f}% VOLATILITY TARGET:")
print(f"   Allocate {w_target[0]*100:.0f}% US / {w_target[1]*100:.0f}% International")
print(f"   Expected: {ret_target*100:.1f}% return")

print("\n4. IF CORRELATION DECREASES (Improved Diversification):")
print(f"   Re-optimize! Lower correlation allows:")
print(f"   - Lower minimum variance: {mvp_vol_fut*100:.2f}% vs {mvp_vol_curr*100:.2f}%")
print(f"   - Higher Sharpe ratio: {max_sharpe_future['sharpe']:.3f} vs {max_sharpe_current['sharpe']:.3f}")

print("\n" + "="*90)

## 5. Minimum Variance Portfolio (N Assets)

### Theory

The **Global Minimum Variance Portfolio (MVP)** is the portfolio with the lowest possible risk. For N assets, we solve:

$$
\begin{align*}
\min_{\mathbf{w}} \quad & \sigma_p^2 = \mathbf{w}^T \boldsymbol{\Sigma} \mathbf{w} \\
\text{subject to} \quad & \mathbf{w}^T \mathbf{1} = 1
\end{align*}
$$

where $\mathbf{1}$ is a vector of ones (budget constraint).

### Key Properties

- **Unique Solution**: Only one MVP exists for a given covariance matrix
- **No Short-Sale Restrictions**: Analytical solution allows negative weights
- **Minimum Risk**: Lies at the leftmost point of the efficient frontier
- **No Return Requirement**: Only minimizes variance without targeting specific return

### Why MVP Matters

1. **Risk-Averse Investors**: Suitable for extremely conservative portfolios
2. **Benchmark**: Starting point for constructing efficient portfolios
3. **Diversification**: Shows maximum risk reduction from diversification
4. **Stability**: Less sensitive to expected return estimates than other portfolios

### Mathematical Formulation

Using Lagrange multipliers, the optimization problem becomes:

$$
\mathcal{L} = \mathbf{w}^T \boldsymbol{\Sigma} \mathbf{w} + \lambda(1 - \mathbf{w}^T \mathbf{1})
$$

Taking the derivative with respect to $\mathbf{w}$ and setting to zero:

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{w}} = 2\boldsymbol{\Sigma} \mathbf{w} - \lambda \mathbf{1} = 0
$$

Solving for $\mathbf{w}$:

$$
\mathbf{w} = \frac{1}{2}\lambda \boldsymbol{\Sigma}^{-1} \mathbf{1}
$$

Applying the budget constraint $\mathbf{w}^T \mathbf{1} = 1$:

$$
\frac{1}{2}\lambda (\boldsymbol{\Sigma}^{-1} \mathbf{1})^T \mathbf{1} = 1 \implies \lambda = \frac{2}{\mathbf{1}^T \boldsymbol{\Sigma}^{-1} \mathbf{1}}
$$

**Final Analytical Solution**:

$$
\mathbf{w}_{\text{MVP}} = \frac{\boldsymbol{\Sigma}^{-1} \mathbf{1}}{\mathbf{1}^T \boldsymbol{\Sigma}^{-1} \mathbf{1}}
$$

The minimum portfolio variance is:

$$
\sigma_{\text{MVP}}^2 = \frac{1}{\mathbf{1}^T \boldsymbol{\Sigma}^{-1} \mathbf{1}}
$$

### Interpretation

- Weights are proportional to $\boldsymbol{\Sigma}^{-1} \mathbf{1}$
- Inverse covariance matrix penalizes correlated assets
- Assets with lower variance and lower correlation receive higher weights
- Solution independent of expected returns

In [ ]:
# Minimum Variance Portfolio Implementation

def minimum_variance_portfolio(cov_matrix):
    """
    Calculate minimum variance portfolio weights analytically.
    
    Parameters
    ----------
    cov_matrix : np.ndarray
        Covariance matrix (N x N)
    
    Returns
    -------
    dict
        Dictionary containing:
        - weights: MVP weights
        - variance: Minimum portfolio variance
        - volatility: Minimum portfolio volatility
    
    Notes
    -----
    Uses analytical formula: w_MVP = Σ^{-1} * 1 / (1^T * Σ^{-1} * 1)
    """
    n = len(cov_matrix)
    ones = np.ones(n)
    
    # Invert covariance matrix
    cov_inv = np.linalg.inv(cov_matrix)
    
    # Calculate weights
    numerator = cov_inv @ ones
    denominator = ones @ cov_inv @ ones
    weights = numerator / denominator
    
    # Calculate variance
    variance = 1 / denominator
    volatility = np.sqrt(variance)
    
    return {
        'weights': weights,
        'variance': variance,
        'volatility': volatility
    }


def minimum_variance_portfolio_optimize(cov_matrix):
    """
    Calculate MVP using scipy.optimize (numerical method).
    
    Parameters
    ----------
    cov_matrix : np.ndarray
        Covariance matrix
    
    Returns
    -------
    dict
        Dictionary with weights, variance, volatility
    """
    n = len(cov_matrix)
    
    # Objective function: portfolio variance
    def objective(w):
        return w @ cov_matrix @ w
    
    # Constraints: weights sum to 1
    constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
    
    # Initial guess: equal weights
    w0 = np.ones(n) / n
    
    # Optimize
    result = optimize.minimize(
        objective,
        w0,
        method='SLSQP',
        constraints=constraints,
        options={'disp': False}
    )
    
    weights = result.x
    variance = result.fun
    volatility = np.sqrt(variance)
    
    return {
        'weights': weights,
        'variance': variance,
        'volatility': volatility,
        'success': result.success
    }


# Example: 5-Asset Portfolio
print("=" * 80)
print("MINIMUM VARIANCE PORTFOLIO: 5-ASSET EXAMPLE")
print("=" * 80)

# Define 5 assets: US Large Cap, US Small Cap, International, Bonds, REITs
asset_names = ['US Large Cap', 'US Small Cap', 'International', 'US Bonds', 'REITs']
expected_returns = np.array([0.10, 0.12, 0.09, 0.04, 0.08])  # 10%, 12%, 9%, 4%, 8%
volatilities = np.array([0.18, 0.25, 0.20, 0.06, 0.22])  # Annualized volatility

# Correlation matrix
correlations = np.array([
    [1.00, 0.80, 0.75, 0.20, 0.60],  # US Large Cap
    [0.80, 1.00, 0.70, 0.15, 0.65],  # US Small Cap
    [0.75, 0.70, 1.00, 0.25, 0.55],  # International
    [0.20, 0.15, 0.25, 1.00, 0.10],  # US Bonds
    [0.60, 0.65, 0.55, 0.10, 1.00]   # REITs
])

# Build covariance matrix
cov_matrix = np.outer(volatilities, volatilities) * correlations

print(f"\nAsset Universe:")
print(f"{'Asset':<20} {'E[Return]':<12} {'Volatility':<12}")
print("-" * 44)
for i, name in enumerate(asset_names):
    print(f"{name:<20} {expected_returns[i]:>11.2%} {volatilities[i]:>11.2%}")

# Calculate MVP - Analytical
mvp_analytical = minimum_variance_portfolio(cov_matrix)

# Calculate MVP - Numerical
mvp_numerical = minimum_variance_portfolio_optimize(cov_matrix)

print(f"\n\n{'='*80}")
print("MINIMUM VARIANCE PORTFOLIO RESULTS")
print("=" * 80)

print(f"\nMethod Comparison:")
print(f"  Analytical Solution:")
print(f"    Volatility: {mvp_analytical['volatility']:.4%}")
print(f"  Numerical Solution:")
print(f"    Volatility: {mvp_numerical['volatility']:.4%}")
print(f"    Optimization Success: {mvp_numerical['success']}")
print(f"  Difference: {abs(mvp_analytical['volatility'] - mvp_numerical['volatility']):.6%}")

print(f"\n\nOptimal Weights (Analytical):")
print(f"{'Asset':<20} {'Weight':<12} {'Contribution':<15}")
print("-" * 47)
for i, name in enumerate(asset_names):
    weight = mvp_analytical['weights'][i]
    print(f"{name:<20} {weight:>11.2%}")

# Portfolio characteristics
mvp_return = expected_returns @ mvp_analytical['weights']
mvp_vol = mvp_analytical['volatility']
mvp_sharpe = (mvp_return - 0.02) / mvp_vol  # Assuming 2% risk-free rate

print(f"\n\nMinimum Variance Portfolio Characteristics:")
print(f"  Expected Return: {mvp_return:.2%}")
print(f"  Volatility: {mvp_vol:.2%}")
print(f"  Sharpe Ratio: {mvp_sharpe:.4f}")

# Compare to equal-weight portfolio
equal_weights = np.ones(5) / 5
equal_return = expected_returns @ equal_weights
equal_vol = np.sqrt(equal_weights @ cov_matrix @ equal_weights)
equal_sharpe = (equal_return - 0.02) / equal_vol

print(f"\n\nComparison to Equal-Weight Portfolio:")
print(f"  Equal-Weight Return: {equal_return:.2%}")
print(f"  Equal-Weight Volatility: {equal_vol:.2%}")
print(f"  Equal-Weight Sharpe: {equal_sharpe:.4f}")
print(f"\n  Risk Reduction from MVP: {(equal_vol - mvp_vol)/equal_vol:.2%}")
print(f"  Return Difference: {mvp_return - equal_return:+.2%}")

print("\n" + "=" * 80)
print('[OK] Minimum Variance Portfolio implementation complete')

In [ ]:
# Visualization: Minimum Variance Portfolio

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: MVP Weight Allocation
ax1.bar(range(len(asset_names)), mvp_analytical['weights'], color='steelblue', edgecolor='navy', linewidth=2)
ax1.set_xticks(range(len(asset_names)))
ax1.set_xticklabels(asset_names, rotation=45, ha='right')
ax1.set_ylabel('Weight', fontsize=12, fontweight='bold')
ax1.set_title('Minimum Variance Portfolio - Optimal Weights', fontsize=13, fontweight='bold')
ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax1.grid(True, alpha=0.3, axis='y')
for i, w in enumerate(mvp_analytical['weights']):
    ax1.text(i, w + 0.01 if w > 0 else w - 0.02, f'{w:.1%}', 
             ha='center', fontsize=10, fontweight='bold')

# Plot 2: Risk Contribution
risk_contributions = []
for i in range(len(asset_names)):
    # Marginal contribution to risk
    marginal_risk = (cov_matrix @ mvp_analytical['weights'])[i]
    risk_contrib = mvp_analytical['weights'][i] * marginal_risk / mvp_analytical['variance']
    risk_contributions.append(risk_contrib)

colors = plt.cm.Set3(range(len(asset_names)))
ax2.pie(risk_contributions, labels=asset_names, autopct='%1.1f%%', startangle=90, colors=colors)
ax2.set_title('Risk Contribution by Asset', fontsize=13, fontweight='bold')

# Plot 3: MVP vs Equal Weight Performance
portfolios_to_compare = ['MVP', 'Equal Weight', '60/40 (Stocks/Bonds)']
returns_compare = [mvp_return, equal_return, 0.60*0.10 + 0.40*0.04]
vols_compare = [mvp_vol, equal_vol, np.sqrt(0.60**2*0.18**2 + 0.40**2*0.06**2 + 2*0.60*0.40*0.18*0.06*0.20)]

x_pos = np.arange(len(portfolios_to_compare))
width = 0.35

bars1 = ax3.bar(x_pos - width/2, [r*100 for r in returns_compare], width, 
                label='Return (%)', color='green', alpha=0.7)
bars2 = ax3.bar(x_pos + width/2, [v*100 for v in vols_compare], width,
                label='Volatility (%)', color='red', alpha=0.7)

ax3.set_ylabel('Percentage (%)', fontsize=12, fontweight='bold')
ax3.set_title('Portfolio Comparison: Return vs Risk', fontsize=13, fontweight='bold')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(portfolios_to_compare, rotation=15, ha='right')
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%', ha='center', va='bottom', fontsize=9)

# Plot 4: Sharpe Ratio Comparison
sharpe_ratios = [mvp_sharpe, equal_sharpe, (returns_compare[2] - 0.02)/vols_compare[2]]
ax4.bar(range(len(portfolios_to_compare)), sharpe_ratios, color=['gold', 'silver', 'bronze'],
        edgecolor='black', linewidth=2)
ax4.set_xticks(range(len(portfolios_to_compare)))
ax4.set_xticklabels(portfolios_to_compare, rotation=15, ha='right')
ax4.set_ylabel('Sharpe Ratio', fontsize=12, fontweight='bold')
ax4.set_title('Risk-Adjusted Performance (Sharpe Ratio)', fontsize=13, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')
for i, sr in enumerate(sharpe_ratios):
    ax4.text(i, sr + 0.02, f'{sr:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print('[OK] MVP visualization complete')

### Practical Example\n\nLet's apply minimum variance portfolio to a real-world scenario...

In [ ]:
# Practical example for Minimum Variance Portfolio

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

## 6. Tangency Portfolio (Maximum Sharpe Ratio)

### Theory

The **Tangency Portfolio** (also called the **Maximum Sharpe Ratio Portfolio**) is the portfolio on the efficient frontier with the highest risk-adjusted return. It represents the optimal risky portfolio when combined with a risk-free asset.

**Optimization Problem**:

$$
\max_{\mathbf{w}} \quad \text{Sharpe Ratio} = \frac{\mathbf{w}^T \boldsymbol{\mu} - r_f}{\sqrt{\mathbf{w}^T \boldsymbol{\Sigma} \mathbf{w}}}
$$

subject to: $\mathbf{w}^T \mathbf{1} = 1$

where $r_f$ is the risk-free rate.

### Capital Allocation Line (CAL)

The tangency portfolio defines the **Capital Allocation Line**:

$$
E[R_p] = r_f + \frac{E[R_{\text{tan}}] - r_f}{\sigma_{\text{tan}}} \cdot \sigma_p
$$

where:
- Slope = Sharpe ratio of tangency portfolio
- All efficient portfolios lie on this line when risk-free borrowing/lending is available
- Investors choose position on CAL based on risk tolerance

### Analytical Solution

The tangency portfolio weights are proportional to:

$$
\mathbf{w}_{\text{tan}} \propto \boldsymbol{\Sigma}^{-1} (\boldsymbol{\mu} - r_f \mathbf{1})
$$

Normalized to sum to 1:

$$
\mathbf{w}_{\text{tan}} = \frac{\boldsymbol{\Sigma}^{-1} (\boldsymbol{\mu} - r_f \mathbf{1})}{\mathbf{1}^T \boldsymbol{\Sigma}^{-1} (\boldsymbol{\mu} - r_f \mathbf{1})}
$$

### Why Tangency Portfolio Matters

1. **Optimal Risky Portfolio**: Best combination of risky assets for any risk-averse investor
2. **Separation Theorem**: Portfolio choice separates into: (1) find tangency portfolio, (2) mix with risk-free asset
3. **CAPM Foundation**: Tangency portfolio is the market portfolio in Capital Asset Pricing Model
4. **Practical Use**: Robo-advisors use tangency portfolio blended with cash/bonds

### Mathematical Formulation\n\nThe mathematical framework for tangency portfolio is given by:\n\n$$\n\\text{Equation placeholder}\n$$\n\nwhere the parameters represent key variables in the model.

In [ ]:
# Tangency Portfolio and Efficient Frontier Implementation

def tangency_portfolio(expected_returns, cov_matrix, risk_free_rate=0.02):
    """
    Calculate tangency (maximum Sharpe ratio) portfolio weights.
    
    Parameters
    ----------
    expected_returns : np.ndarray
        Expected returns vector
    cov_matrix : np.ndarray
        Covariance matrix
    risk_free_rate : float
        Risk-free rate (default 2%)
    
    Returns
    -------
    dict
        Dictionary with weights, return, volatility, Sharpe ratio
    """
    n = len(expected_returns)
    ones = np.ones(n)
    
    # Analytical solution
    cov_inv = np.linalg.inv(cov_matrix)
    excess_returns = expected_returns - risk_free_rate
    
    numerator = cov_inv @ excess_returns
    denominator = ones @ cov_inv @ excess_returns
    weights = numerator / denominator
    
    # Portfolio characteristics
    port_return = weights @ expected_returns
    port_variance = weights @ cov_matrix @ weights
    port_volatility = np.sqrt(port_variance)
    sharpe_ratio = (port_return - risk_free_rate) / port_volatility
    
    return {
        'weights': weights,
        'return': port_return,
        'volatility': port_volatility,
        'sharpe_ratio': sharpe_ratio
    }


def efficient_frontier(expected_returns, cov_matrix, num_portfolios=100, risk_free_rate=0.02):
    """
    Generate efficient frontier by varying target returns.
    
    Parameters
    ----------
    expected_returns : np.ndarray
        Expected returns vector
    cov_matrix : np.ndarray
        Covariance matrix
    num_portfolios : int
        Number of portfolios on frontier
    risk_free_rate : float
        Risk-free rate
    
    Returns
    -------
    dict
        Dictionary with returns, volatilities, Sharpe ratios, and weights
    """
    n = len(expected_returns)
    
    # Get MVP and tangency portfolio to determine range
    mvp = minimum_variance_portfolio(cov_matrix)
    tan = tangency_portfolio(expected_returns, cov_matrix, risk_free_rate)
    
    mvp_return = expected_returns @ mvp['weights']
    max_return = expected_returns.max() * 1.5  # Extend beyond max asset return
    
    # Target returns from MVP to beyond tangency
    target_returns = np.linspace(mvp_return, max_return, num_portfolios)
    
    frontier_returns = []
    frontier_vols = []
    frontier_sharpes = []
    frontier_weights = []
    
    for target_return in target_returns:
        # Optimize: minimize variance subject to target return
        def objective(w):
            return w @ cov_matrix @ w
        
        constraints = [
            {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},  # Weights sum to 1
            {'type': 'eq', 'fun': lambda w: w @ expected_returns - target_return}  # Target return
        ]
        
        w0 = np.ones(n) / n
        result = optimize.minimize(objective, w0, method='SLSQP', constraints=constraints)
        
        if result.success:
            w = result.x
            vol = np.sqrt(w @ cov_matrix @ w)
            ret = w @ expected_returns
            sharpe = (ret - risk_free_rate) / vol
            
            frontier_returns.append(ret)
            frontier_vols.append(vol)
            frontier_sharpes.append(sharpe)
            frontier_weights.append(w)
    
    return {
        'returns': np.array(frontier_returns),
        'volatilities': np.array(frontier_vols),
        'sharpe_ratios': np.array(frontier_sharpes),
        'weights': np.array(frontier_weights),
        'mvp': mvp,
        'tangency': tan
    }


# Calculate all key portfolios
print("=" * 90)
print("COMPREHENSIVE PORTFOLIO OPTIMIZATION")
print("=" * 90)

rf_rate = 0.02  # 2% risk-free rate

# Minimum Variance Portfolio
mvp = minimum_variance_portfolio(cov_matrix)
mvp_return = expected_returns @ mvp['weights']
mvp_sharpe = (mvp_return - rf_rate) / mvp['volatility']

# Tangency Portfolio (Maximum Sharpe Ratio)
tangency = tangency_portfolio(expected_returns, cov_matrix, rf_rate)

# Efficient Frontier
frontier = efficient_frontier(expected_returns, cov_matrix, 50, rf_rate)

print(f"\n{'='*90}")
print("KEY PORTFOLIO SOLUTIONS")
print("=" * 90)

print(f"\n1. MINIMUM VARIANCE PORTFOLIO (MVP)")
print(f"   Return: {mvp_return:.2%}")
print(f"   Volatility: {mvp['volatility']:.2%}")
print(f"   Sharpe Ratio: {mvp_sharpe:.4f}")
print(f"   Weights:")
for i, name in enumerate(asset_names):
    print(f"     {name:<20}: {mvp['weights'][i]:>7.2%}")

print(f"\n2. TANGENCY PORTFOLIO (Maximum Sharpe Ratio)")
print(f"   Return: {tangency['return']:.2%}")
print(f"   Volatility: {tangency['volatility']:.2%}")
print(f"   Sharpe Ratio: {tangency['sharpe_ratio']:.4f}")
print(f"   Weights:")
for i, name in enumerate(asset_names):
    print(f"     {name:<20}: {tangency['weights'][i]:>7.2%}")

print(f"\n3. EFFICIENT FRONTIER CHARACTERISTICS")
print(f"   Number of portfolios: {len(frontier['returns'])}")
print(f"   Return range: {frontier['returns'].min():.2%} to {frontier['returns'].max():.2%}")
print(f"   Risk range: {frontier['volatilities'].min():.2%} to {frontier['volatilities'].max():.2%}")
print(f"   Max Sharpe on frontier: {frontier['sharpe_ratios'].max():.4f}")

# Comparison table
print(f"\n\n{'='*90}")
print("PORTFOLIO COMPARISON TABLE")
print("=" * 90)
print(f"{'Portfolio':<25} {'Return':<12} {'Volatility':<12} {'Sharpe':<10}")
print("-" * 59)
print(f"{'MVP':<25} {mvp_return:>11.2%} {mvp['volatility']:>11.2%} {mvp_sharpe:>9.4f}")
print(f"{'Tangency':<25} {tangency['return']:>11.2%} {tangency['volatility']:>11.2%} {tangency['sharpe_ratio']:>9.4f}")
print(f"{'Equal Weight':<25} {equal_return:>11.2%} {equal_vol:>11.2%} {equal_sharpe:>9.4f}")
print(f"{'60/40 (Stocks/Bonds)':<25} {0.60*0.10+0.40*0.04:>11.2%} {vols_compare[2]:>11.2%} {sharpe_ratios[2]:>9.4f}")

print("\n" + "=" * 90)
print('[OK] Tangency Portfolio and Efficient Frontier implementation complete')

In [ ]:
# Visualization: Complete Efficient Frontier with CAL

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 13))

# Plot 1: Efficient Frontier with Individual Assets and CAL
ax1.plot(frontier['volatilities'], frontier['returns'], 'b-', linewidth=3, label='Efficient Frontier')
ax1.scatter([volatilities[i] for i in range(len(asset_names))], 
           expected_returns, s=200, c='red', marker='D', edgecolors='darkred', 
           linewidth=2, label='Individual Assets', zorder=5)

# Add labels for individual assets
for i, name in enumerate(asset_names):
    ax1.annotate(name, (volatilities[i], expected_returns[i]), 
                xytext=(5, 5), textcoords='offset points', fontsize=9)

# Plot MVP and Tangency
ax1.scatter(mvp['volatility'], mvp_return, s=300, c='gold', marker='*', 
           edgecolors='orange', linewidth=2, label='MVP', zorder=6)
ax1.scatter(tangency['volatility'], tangency['return'], s=300, c='lime', marker='*',
           edgecolors='green', linewidth=2, label='Tangency (Max Sharpe)', zorder=6)

# Capital Allocation Line
cal_vols = np.linspace(0, max(frontier['volatilities']) * 1.1, 100)
cal_returns = rf_rate + tangency['sharpe_ratio'] * cal_vols
ax1.plot(cal_vols, cal_returns, 'g--', linewidth=2, label='Capital Allocation Line', alpha=0.7)
ax1.scatter(0, rf_rate, s=200, c='purple', marker='s', edgecolors='indigo',
           linewidth=2, label='Risk-Free Asset', zorder=6)

ax1.set_xlabel('Volatility (Risk)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Expected Return', fontsize=12, fontweight='bold')
ax1.set_title('Efficient Frontier and Capital Allocation Line', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right', fontsize=9)
ax1.grid(True, alpha=0.3)

# Plot 2: Sharpe Ratio along Frontier
ax2.plot(frontier['volatilities'], frontier['sharpe_ratios'], 'purple', linewidth=2)
ax2.axhline(y=tangency['sharpe_ratio'], color='green', linestyle='--', linewidth=2,
           label=f'Max Sharpe = {tangency["sharpe_ratio"]:.4f}')
ax2.scatter(tangency['volatility'], tangency['sharpe_ratio'], s=300, c='lime',
           marker='*', edgecolors='green', linewidth=2, zorder=5)
ax2.set_xlabel('Portfolio Volatility', fontsize=12, fontweight='bold')
ax2.set_ylabel('Sharpe Ratio', fontsize=12, fontweight='bold')
ax2.set_title('Sharpe Ratio Along Efficient Frontier', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# Plot 3: Weight Evolution along Frontier (stacked area)
# Select subset of frontier portfolios for clarity
step = max(1, len(frontier['weights']) // 20)
weights_subset = frontier['weights'][::step]
returns_subset = frontier['returns'][::step]

weights_df = pd.DataFrame(weights_subset, columns=asset_names)
weights_df['Return'] = returns_subset

ax3.stackplot(range(len(weights_subset)), 
             *[weights_df[name] for name in asset_names],
             labels=asset_names, alpha=0.8)
ax3.set_xlabel('Portfolio Index (Lower to Higher Return)', fontsize=12, fontweight='bold')
ax3.set_ylabel('Weight', fontsize=12, fontweight='bold')
ax3.set_title('Asset Allocation Along Efficient Frontier', fontsize=14, fontweight='bold')
ax3.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=9)
ax3.set_ylim([0, 1])
ax3.grid(True, alpha=0.3, axis='y')

# Plot 4: Tangency vs MVP Weights Comparison
x = np.arange(len(asset_names))
width = 0.35

bars1 = ax4.bar(x - width/2, mvp['weights'], width, label='MVP', color='gold', alpha=0.8, edgecolor='orange', linewidth=2)
bars2 = ax4.bar(x + width/2, tangency['weights'], width, label='Tangency', color='lime', alpha=0.8, edgecolor='green', linewidth=2)

ax4.set_ylabel('Weight', fontsize=12, fontweight='bold')
ax4.set_title('MVP vs Tangency Portfolio Weights', fontsize=14, fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(asset_names, rotation=45, ha='right')
ax4.legend(fontsize=11)
ax4.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax4.grid(True, alpha=0.3, axis='y')

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if abs(height) > 0.01:  # Only label significant weights
            ax4.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.1%}', ha='center', 
                    va='bottom' if height > 0 else 'top', fontsize=8)

plt.tight_layout()
plt.show()

print('[OK] Efficient Frontier visualization complete')

### Practical Example\n\nLet's apply tangency portfolio to a real-world scenario...

In [ ]:
# Practical example for Tangency Portfolio

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

## 7. Two-Fund Separation Theorem\n\n### Theory\n\nDetailed explanation of two-fund separation theorem...

### Mathematical Formulation\n\nThe mathematical framework for two-fund separation theorem is given by:\n\n$$\n\\text{Equation placeholder}\n$$\n\nwhere the parameters represent key variables in the model.

In [ ]:
# Python implementation for Two-Fund Separation Theorem

def compute_two_fund_separation_theorem():
    """
    Two-Fund Separation Theorem implementation
    """
    # Implementation code here
    pass

print(f'[OK] Two-Fund Separation Theorem implementation complete')

In [ ]:
# Visualization for Two-Fund Separation Theorem

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('Two-Fund Separation Theorem')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Practical Example\n\nLet's apply two-fund separation theorem to a real-world scenario...

In [ ]:
# Practical example for Two-Fund Separation Theorem

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

## Comprehensive Case Study\n\nNow let's combine all the concepts in a comprehensive example...

In [ ]:
# Comprehensive case study implementation

class CaseStudy:
    def __init__(self):
        self.data = None
        self.results = {}
    
    def run_analysis(self):
        """Run complete analysis"""
        pass

# Execute case study
study = CaseStudy()
print('[OK] Case study initialized')

## Practice Exercises\n\nTest your understanding with these exercises:\n\n### Exercise 1\nDescription of exercise 1...\n\n### Exercise 2\nDescription of exercise 2...\n\n### Exercise 3\nDescription of exercise 3...

In [ ]:
# Your solution for Exercise 1



## Summary and Key Takeaways

### What We Learned

This notebook provided a comprehensive treatment of mean-variance optimization and Modern Portfolio Theory. Here are the key takeaways:

#### 1. Core Concepts
- **Portfolio return** is the weighted average of asset returns: $E[R_p] = \mathbf{w}^T \boldsymbol{\mu}$
- **Portfolio risk** depends on covariances, not just individual volatilities: $\sigma_p^2 = \mathbf{w}^T \boldsymbol{\Sigma} \mathbf{w}$
- **Diversification** reduces risk when assets are not perfectly correlated
- **Efficient frontier** shows maximum return for each risk level

#### 2. Key Portfolio Solutions
- **Minimum Variance Portfolio (MVP)**: Lowest risk portfolio, independent of expected returns
- **Tangency Portfolio**: Maximum Sharpe ratio, optimal risky portfolio when risk-free asset exists
- **Capital Allocation Line (CAL)**: Combines tangency portfolio with risk-free asset
- **Two-Fund Separation**: Any efficient portfolio can be formed from MVP and tangency portfolio

#### 3. Mathematical Framework
- **Analytical solutions** exist for 2-asset case and special N-asset cases
- **Lagrange multipliers** solve constrained optimization problems
- **Matrix inversion** ($\boldsymbol{\Sigma}^{-1}$) required for optimal weights
- **Quadratic programming** generalizes to constraints (long-only, bounds, etc.)

### When to Use Mean-Variance Optimization

**Use MVO When:**
- Constructing strategic asset allocation for long-term portfolios
- Need mathematical framework for diversification
- Have reliable estimates of expected returns and covariances
- Working with liquid, exchange-traded assets
- Building model portfolios or robo-advisor allocations

**Exercise Caution When:**
- Expected return estimates are unstable (estimation error)
- Historical data doesn't reflect future distributions
- Assets have non-normal return distributions (fat tails, skewness)
- Transaction costs and taxes are significant
- Leverage constraints or other practical constraints apply

### Advantages of MVO

1. **Rigorous Framework**: Mathematical foundation for portfolio construction
2. **Diversification Benefit**: Explicitly captures correlation effects
3. **Optimal Solutions**: Provides globally optimal portfolios under assumptions
4. **Risk Management**: Quantifies portfolio risk accurately
5. **Industry Standard**: Widely adopted and understood by professionals

### Limitations and Criticisms

1. **Estimation Error**: Small changes in inputs lead to large changes in optimal weights
   - **Solution**: Use robust optimization, Bayesian methods, or resampling

2. **Normal Returns Assumption**: Real returns have fat tails and are non-normal
   - **Solution**: Use downside risk measures (CVaR, semivariance)

3. **Static Optimization**: Doesn't account for changing market conditions
   - **Solution**: Dynamic asset allocation, regime-switching models

4. **Ignores Higher Moments**: Only considers mean and variance, not skewness/kurtosis
   - **Solution**: Mean-variance-skewness optimization

5. **Unrealistic Constraints**: Allows extreme leverage and short positions
   - **Solution**: Add practical constraints (long-only, position limits)

### Extensions and Advanced Topics

**Black-Litterman Model** (1992):
- Combines market equilibrium with investor views
- Reduces estimation error from expected returns
- Provides more stable, intuitive portfolios

**Robust Optimization**:
- Accounts for parameter uncertainty
- Worst-case optimization over uncertainty sets
- Less sensitive to estimation error

**Risk Parity**:
- Equal risk contribution from all assets
- Extension of MVP concept
- Popular in institutional portfolios

**Factor Models**:
- Covariance matrix from factor exposures
- Reduces dimensionality and estimation error
- Barra, Axioma factor models

**Transaction Cost Optimization**:
- Incorporates trading costs explicitly
- Trade-off between optimality and turnover
- Important for practical implementation

### Academic References

1. **Markowitz, H. (1952)**. "Portfolio Selection." *Journal of Finance*, 7(1), 77-91.
   - Original mean-variance optimization paper, Nobel Prize 1990

2. **Sharpe, W.F. (1964)**. "Capital Asset Prices: A Theory of Market Equilibrium." *Journal of Finance*, 19(3), 425-442.
   - CAPM and tangency portfolio theory

3. **Black, F., and Litterman, R. (1992)**. "Global Portfolio Optimization." *Financial Analysts Journal*, 48(5), 28-43.
   - Black-Litterman model combining views with equilibrium

4. **Merton, R.C. (1972)**. "An Analytic Derivation of the Efficient Portfolio Frontier." *Journal of Financial and Quantitative Analysis*, 7(4), 1851-1872.
   - Mathematical foundations and continuous-time extensions

5. **Michaud, R.O. (1998)**. *Efficient Asset Management*. Harvard Business School Press.
   - Resampled efficiency, practical implementation

6. **Best, M.J., and Grauer, R.R. (1991)**. "On the Sensitivity of Mean-Variance-Efficient Portfolios." *Review of Financial Studies*, 4(2), 315-342.
   - Estimation error and sensitivity analysis

### Practitioner Resources

- **Hull, J.C. (2022)**. *Options, Futures, and Other Derivatives*, 11th Edition. Pearson.
- **Bodie, Z., Kane, A., and Marcus, A.J. (2020)**. *Investments*, 12th Edition. McGraw-Hill.
- **Grinold, R.C., and Kahn, R.N. (1999)**. *Active Portfolio Management*, 2nd Edition. McGraw-Hill.

### Next Steps in Your Learning Journey

1. **Implement MVO** with real market data (use yfinance or Alpha Vantage APIs)
2. **Test sensitivity** of optimal weights to input changes
3. **Add constraints** (long-only, sector limits, turnover constraints) using cvxpy
4. **Backtest** optimized portfolios vs benchmarks (equal-weight, 60/40, market-cap)
5. **Explore Black-Litterman** model for incorporating views
6. **Study factor models** (Fama-French) for covariance estimation
7. **Learn CVaR optimization** for downside risk management

### Congratulations!

You have completed a comprehensive study of mean-variance portfolio optimization. You now have the tools to:
- Construct efficient portfolios mathematically
- Understand risk-return trade-offs quantitatively
- Implement optimization algorithms in Python
- Apply constraints for practical portfolios
- Evaluate portfolio performance using Sharpe ratios

This is the foundation for advanced portfolio management and quantitative asset allocation!

---

**Notebook Complete**: Mean-Variance Optimization
**Date**: December 2, 2025
**Estimated Engagement Time**: 90-120 minutes
**Level**: Intermediate